# VadCLIP V18 · Biến thiên CLIP và mẫu âm khó trên nền V7

V18 kiểm tra liệu biến thiên embedding theo thời gian và học với mẫu âm có hình ảnh tương tự có giúp phân biệt hành vi bất thường trong bối cảnh dễ nhầm hay không. Nhánh hiệu chỉnh nhận feature CLIP hiện tại và chênh lệch có dấu với các snippet trước và sau, cách 1, 2 và 4 bước. Tín hiệu được tính trên từng video, bao gồm cả các cặp qua ranh giới block V7. Vị trí thiếu hàng xóm đóng góp 0.

| Cấu hình | Tín hiệu | Cách ghép mẫu âm |
|---|---|---|
| V7 | Checkpoint VadCLIP và cửa sổ chồng lấn | Đối chứng |
| CLIP | Feature hiện tại | Ngẫu nhiên |
| CLIP + temporal | Feature hiện tại và chênh lệch embedding | Ngẫu nhiên |
| V18 | Feature hiện tại và chênh lệch embedding | Độ tương đồng cosine cao nhất trong nhóm mẫu âm của batch |

Ba nhánh học dùng chung split theo nhóm nguồn, khởi tạo và lịch batch. Các đoạn dương được sàng lọc bằng teacher V7 và sự đồng thuận với snippet lân cận. Mẫu âm đến từ video Normal. Hệ số residual được chọn theo AP cấp video trên development. UCF hiệu chỉnh logit nhánh C, XD hiệu chỉnh logit Normal của nhánh A.

## Sáu input Kaggle

| Dataset | Dữ liệu |
|---|---|
| VadCLIP UCF-Crime Test290 | Feature CLIP test crop 5 |
| VadCLIP UCF Checkpoint | model_ucf.pth |
| VadCLIP XD-Violence Test 800 | Feature CLIP test và manifest gốc |
| VadCLIP XD-Violence Checkpoint | model_xd.pth |
| VadCLIP UCF-Crime Train1610 V5 | vadclip_ucf_train_v5.zip |
| VadCLIP XD-Violence Train3954 V5 | vadclip_xd_train_v5.zip |

## Cách chạy

1. Gắn sáu input, chọn GPU T4 và bật Internet để nạp mã VadCLIP và trọng số CLIP.
2. Giữ RUN_MODE = "smoke" và chọn Run All. Smoke chạy hai batch train mỗi tập và kiểm tra cả bốn cấu hình.
3. Khi hai tập báo ok, đặt RUN_MODE = "full" và chạy lại.
4. Tải /kaggle/working/vadclip_v18/run_*/vadclip_v18_results.zip. Gói gồm bảng comparison.md và comparison.csv ở chế độ full, summary.json, log và checkpoint residual. Bật INCLUDE_PREDICTIONS để lưu điểm theo snippet.

Đọc hàng V18 so với V7 cho frame ROC-AUC trên UCF-Crime và frame AP trên XD-Violence. So CLIP + temporal với CLIP để đo hiệu ứng biến thiên thời gian, rồi so V18 với CLIP + temporal để đo hiệu ứng mẫu âm khó. Xem thêm tỷ lệ báo động giả ở các mục tiêu recall 50% và 80%, recall thực tế sau khi xét điểm trùng, và recall sự kiện ở cùng trần FPR. Các chỉ số theo ngưỡng trên test dùng cho phân tích hồi cứu. Chênh lệch embedding có thể phản ứng với chuyển cảnh và chỉ phản ánh tín hiệu đã được giữ trong feature CLIP.


In [ ]:
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
RUN_MODE = "smoke"
DATASETS = ("ucf", "xd")
UCF_DATA_ROOT = None
UCF_CHECKPOINT = None
XD_DATA_ROOT = None
XD_FEATURE_ZIP = None
XD_CHECKPOINT = None
CLIP_WEIGHTS = None

# ruff: noqa: F821
UCF_TRAIN_ROOT = None
XD_TRAIN_ROOT = None
SEED = 42
HALF_BATCH = 8
TEMPORAL_CONFIG = {"epochs": 3}
BOOTSTRAP_REPLICATES = 500
INCLUDE_PREDICTIONS = False
TRAIN_MANIFESTS = {
    "ucf": "08503ee0733dc34ca9b955ad8d7656d4de70fbe4903f1a716ca6aff56860683f",
    "xd": "156c0638d4668c4ef7896216108981e6d14cc582637ddad744a22db3718cf157",
}


In [ ]:
SOURCES = {'v7_windows.py': '\nimport numpy as np\nimport torch\n\n\ndef window_plan(steps, width):\n    # Map each valid snippet to exactly one overlapping window.\n    if width < 2 or steps <= width:\n        raise ValueError("Window plan requires a multi-block video.")\n    stride = width // 2\n    starts = list(range(0, steps - width, stride))\n    starts.append(starts[-1] + stride)\n    owner = np.full(steps, -1, dtype=np.int64)\n    quality = np.full(steps, -1, dtype=np.int64)\n    for index, start in enumerate(starts):\n        stop = min(start + width, steps)\n        local = np.arange(stop - start)\n        left = local if start else width\n        right = stop - start - 1 - local if stop < steps else width\n        context = np.minimum(left, right)\n        better = context > quality[start:stop]\n        quality[start:stop][better] = context[better]\n        owner[start:stop][better] = index\n    if (owner < 0).any():\n        raise RuntimeError("Window plan missed valid snippets.")\n    return [(start, np.flatnonzero(owner[start:min(start + width, steps)] == i))\n            for i, start in enumerate(starts)]\n\n\ndef overlap_forward(forward, images, mask, text, lengths, make_mask):\n    # The upstream loader passes one video as [blocks, snippets, channels].\n    sizes = torch.as_tensor(lengths).detach().cpu().reshape(-1)\n    blocks, width, channels = images.shape\n    if sizes.numel() != blocks or (sizes < 0).any() or (sizes > width).any():\n        raise ValueError("Unexpected VadCLIP block lengths.")\n    steps = int(sizes.sum())\n    if steps <= width:\n        return forward(images, mask, text, lengths)\n    expected = (steps - torch.arange(blocks) * width).clamp(min=0, max=width)\n    if not torch.equal(sizes, expected):\n        raise ValueError("VadCLIP video blocks are not a contiguous padded sequence.")\n\n    flat = images.reshape(-1, channels)[:steps]\n    plan = window_plan(steps, width)\n    for offset in range(0, len(plan), 8):\n        chunk = plan[offset:offset + 8]\n        batch = images.new_zeros((len(chunk), width, channels))\n        valid_lengths = torch.tensor([min(width, steps - start) for start, _ in chunk])\n        for row, ((start, _), length) in enumerate(zip(chunk, valid_lengths.tolist(), strict=True)):\n            batch[row, :length] = flat[start:start + length]\n        batch_mask = make_mask(valid_lengths, width).to(images.device)\n        text_features, logits_c, logits_a = forward(batch, batch_mask, text, valid_lengths)\n        if offset == 0:\n            result_c = logits_c.new_zeros((blocks * width, *logits_c.shape[2:]))\n            result_a = logits_a.new_zeros((blocks * width, *logits_a.shape[2:]))\n        for row, (start, positions) in enumerate(chunk):\n            indices = torch.as_tensor(positions, device=images.device)\n            result_c[start + indices] = logits_c[row, indices]\n            result_a[start + indices] = logits_a[row, indices]\n    return (text_features, result_c.reshape(blocks, width, *result_c.shape[1:]),\n            result_a.reshape(blocks, width, *result_a.shape[1:]))\n', 'head.py': '"""Residual scoring from CLIP appearance and multi-scale temporal differences."""\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nTEMPORAL_LAGS = (1, 2, 4)\n\n\ndef anomaly_logits(c, a):\n    a = a.float()\n    return c.float().squeeze(-1), a[..., 1:].logsumexp(-1) - a[..., 0]\n\n\nclass TemporalHead(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.clip = nn.Linear(512, 64)\n        self.temporal = nn.ModuleList(\n            nn.Linear(512, 128, bias=False) for _ in TEMPORAL_LAGS\n        )\n        self.output = nn.Sequential(nn.LayerNorm(64), nn.GELU(), nn.Linear(64, 1))\n        nn.init.zeros_(self.output[-1].weight)\n        nn.init.zeros_(self.output[-1].bias)\n\n    def forward(self, clip, use_temporal):\n        # Use the same feature precision as the training cache.\n        features = F.normalize(clip.to(torch.float16).float(), dim=-1) * (512 ** 0.5)\n        visual = self.clip(features)\n        if use_temporal:\n            for lag, projection in zip(TEMPORAL_LAGS, self.temporal, strict=True):\n                if len(features) <= lag:\n                    continue\n                # Linear projection commutes with subtraction. Each direction has\n                # its own weights, and missing neighbors contribute zero.\n                past, future = projection(features).chunk(2, dim=-1)\n                past = F.pad(past[lag:] - past[:-lag], (0, 0, lag, 0))\n                future = F.pad(future[:-lag] - future[lag:], (0, 0, 0, lag))\n                visual = visual + (past + future) / len(TEMPORAL_LAGS)\n        return 2 * self.output(visual).squeeze(-1).tanh()\n\n\ndef load_head(path, strength, use_temporal, device):\n    head = TemporalHead().to(device)\n    head.load_state_dict(torch.load(path, map_location=device, weights_only=True), strict=True)\n    return head.eval(), strength, use_temporal\n\n\n@torch.no_grad()\ndef apply_head(bundle, clip, lengths, c, a, primary):\n    if bundle is None:\n        return c, a\n    head, strength, use_temporal = bundle\n    if strength == 0:\n        return c, a\n    steps = int(torch.as_tensor(lengths).sum())\n    delta = head(clip.reshape(-1, 512)[:steps], use_temporal) * strength\n    if not torch.isfinite(delta).all():\n        raise ValueError("Non-finite V18 residual")\n    delta = F.pad(delta, (0, clip.shape[0] * clip.shape[1] - steps)).reshape(clip.shape[:2])\n    if primary == "C":\n        return c + delta[..., None].to(c.dtype), a\n    modified = a.clone()\n    modified[..., 0] -= delta.to(a.dtype)\n    return c, modified\n', 'analysis.py': '"""Paired, video-bootstrap metrics without treating correlated frames as iid."""\n\nimport numpy as np\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\n\ndef ranked_counts(scores, positives, negatives):\n    order = np.argsort(-scores, kind="stable")\n    end = np.r_[np.flatnonzero(np.diff(scores[order]) != 0), len(order) - 1]\n    return order, end, positives[order], negatives[order]\n\n\ndef weighted_metrics(ranked, video_weights):\n    order, end, pos, neg = ranked\n    weights = video_weights[order]\n    tp = np.cumsum(pos * weights)[end]\n    fp = np.cumsum(neg * weights)[end]\n    if not len(tp) or tp[-1] <= 0 or fp[-1] <= 0:\n        return None\n    recall = tp / tp[-1]\n    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=tp + fp > 0)\n    ap = np.sum(np.diff(np.r_[0, recall]) * precision)\n    x, y = np.r_[0, fp / fp[-1]], np.r_[0, recall]\n    auc = np.sum(np.diff(x) * (y[1:] + y[:-1]) / 2)\n    return float(auc), float(ap)\n\n\ndef paired_bootstrap(scores0, scores1, gt, offsets, repeats, seed=42):\n    positives = gt.reshape(-1, 16).sum(1).astype(float)\n    negatives = 16 - positives\n    ranks = [ranked_counts(s, positives, negatives) for s in (scores0, scores1)]\n    video_index = np.repeat(np.arange(len(offsets) - 1), np.diff(offsets))\n    rng = np.random.default_rng(seed)\n    deltas = []\n    for _ in range(repeats):\n        counts = rng.multinomial(len(offsets) - 1, np.full(len(offsets) - 1, 1 / (len(offsets) - 1)))\n        values = [weighted_metrics(r, counts[video_index]) for r in ranks]\n        if all(v is not None for v in values):\n            deltas.append(np.subtract(values[1], values[0]) * 100)\n    if not deltas:\n        return {"requested": repeats, "valid": 0, "auc_ci95_pp": None, "ap_ci95_pp": None}\n    bounds = np.quantile(deltas, [0.025, 0.975], axis=0)\n    return {"requested": repeats, "valid": len(deltas),\n            "auc_ci95_pp": bounds[:, 0].tolist(), "ap_ci95_pp": bounds[:, 1].tolist()}\n\n\ndef false_alarm_diagnostics(scores, gt, offsets):\n    """Retrospective test-label operating points, NOT deployable calibrated thresholds."""\n    frames = np.repeat(scores, 16)\n    truth = gt.astype(bool)\n    positive_video = np.zeros(len(gt), dtype=bool)\n    for a, b in zip(offsets[:-1] * 16, offsets[1:] * 16, strict=True):\n        positive_video[a:b] = truth[a:b].any()\n    result = []\n    for requested in (0.5, 0.8):\n        positive_scores = np.sort(frames[truth])[::-1]\n        threshold = float(positive_scores[max(0, int(np.ceil(requested * len(positive_scores))) - 1)])\n        detected = frames >= threshold\n        row = {"requested_recall": requested, "actual_recall": float(detected[truth].mean()),\n               "threshold": threshold, "status": "retrospective_test_diagnostic_only"}\n        for name, mask in (("all_background", ~truth),\n                           ("background_in_positive_videos", ~truth & positive_video),\n                           ("all_negative_videos", ~truth & ~positive_video)):\n            row[name + "_frames"] = int(mask.sum())\n            row[name + "_fp"] = int((detected & mask).sum())\n            row[name + "_fpr"] = float(detected[mask].mean()) if mask.any() else None\n        result.append(row)\n    return result\n\n\ndef score_arrays(scores, gt):\n    x = np.repeat(scores, 16)\n    return {"roc_auc": float(roc_auc_score(gt, x)),\n            "ap": float(average_precision_score(gt, x))}\n\n\nimport math\n\ndef event_intervals(gt, frame_offsets):\n    spans = []\n    owners = []\n    for video, (start, stop) in enumerate(zip(frame_offsets[:-1], frame_offsets[1:], strict=True)):\n        segment = gt[start:stop].astype(bool, copy=False)\n        edges = np.flatnonzero(np.diff(np.r_[False, segment, False])) + start\n        intervals = edges.reshape(-1, 2)\n        spans.extend(intervals.tolist())\n        owners.extend([video] * len(intervals))\n    return np.asarray(spans, dtype=np.int64).reshape(-1, 2), np.asarray(owners, dtype=np.int64)\n\n\ndef operating_point(gt, scores, negative, abnormal_background, spans, fpr):\n    false_budget = math.floor(fpr * len(negative))\n    split = len(negative) - false_budget - 1\n    cutoff = float(np.partition(negative, split)[split])\n    # A strict comparison excludes the entire tie at the cutoff.\n    detected = scores > cutoff\n    count = np.cumsum(np.r_[0, detected])\n    lengths = spans[:, 1] - spans[:, 0]\n    recalls = (count[spans[:, 1]] - count[spans[:, 0]]) / lengths\n    return {\n        "exclusive_score_cutoff_offline": cutoff,\n        "realized_background_fpr": float(detected[~gt].mean()),\n        "abnormal_video_background_fpr": (\n            float(detected[abnormal_background].mean()) if abnormal_background.any() else None\n        ),\n        "mean_event_frame_recall": float(recalls.mean()),\n        "mean_short_event_frame_recall": float(recalls[lengths <= 48].mean())\n        if (lengths <= 48).any() else None,\n    }, recalls\n\n\ndef paired_event_ci(differences, owners, videos, seed, replicates):\n    if replicates == 0:\n        return None\n    totals = np.bincount(owners, weights=differences, minlength=videos)\n    events = np.bincount(owners, minlength=videos)\n    rng = np.random.default_rng(seed)\n    samples = rng.integers(videos, size=(replicates, videos))\n    counts = events[samples].sum(axis=1)\n    usable = counts > 0\n    values = totals[samples[usable]].sum(axis=1) / counts[usable]\n    if len(values) == 0:\n        return None\n    return {\n        "method": "paired_video_percentile_fixed_offline_cutoffs",\n        "replicates": int(usable.sum()),\n        "95pct": np.quantile(values, [0.025, 0.975]).tolist(),\n    }\n\n\ndef paired_benchmark_ci(gt, snippet_scores, offsets, frames, key, expected, seed, replicates):\n    """Bootstrap videos with replacement, retaining exact frame counts and score ties."""\n    if replicates == 0:\n        return None\n    owners = np.repeat(np.arange(len(offsets) - 1), np.diff(offsets))\n    pos = gt.reshape(-1, frames).sum(axis=1)\n    neg = frames - pos\n\n    def prepare(scores):\n        order = np.argsort(-scores, kind="stable")\n        starts = np.r_[0, 1 + np.flatnonzero(np.diff(scores[order]) != 0)]\n        return owners[order], pos[order], neg[order], starts\n\n    def evaluate(counts, arrays):\n        video, positive, negative, starts = arrays\n        weights = counts[video]\n        true = np.add.reduceat(positive * weights, starts)\n        false = np.add.reduceat(negative * weights, starts)\n        total_true, total_false = true.sum(), false.sum()\n        if total_true == 0 or total_false == 0:\n            return np.nan\n        if key == "roc_auc":\n            below = total_false - np.cumsum(false)\n            return float(np.sum(true * (below + 0.5 * false)) / (total_true * total_false))\n        true_cum = np.cumsum(true)\n        seen = true_cum + np.cumsum(false)\n        precision = np.divide(true_cum, seen, out=np.zeros_like(seen, dtype=float), where=seen > 0)\n        return float(np.sum(true * precision) / total_true)\n\n    left, right = [prepare(scores) for scores in snippet_scores]\n    videos = len(offsets) - 1\n    counts = np.ones(videos, dtype=np.int64)\n    for arrays, target in zip((left, right), expected, strict=True):\n        if not math.isclose(evaluate(counts, arrays), target, abs_tol=1e-9):\n            raise ValueError("Metric bootstrap trên toàn tập lệch evaluator gốc.")\n    rng = np.random.default_rng(seed)\n    differences = []\n    for _ in range(replicates):\n        counts = np.bincount(rng.integers(videos, size=videos), minlength=videos)\n        delta = evaluate(counts, right) - evaluate(counts, left)\n        if np.isfinite(delta):\n            differences.append(delta)\n    return {\n        "method": "paired_video_percentile",\n        "replicates": len(differences),\n        "95pct": np.quantile(differences, [0.025, 0.975]).tolist() if differences else None,\n    }\n\n\ndef compare_coverage(prefix, paths, reports, fprs, seed, bootstrap, reference, variant):\n    control, treatment = reports[reference], reports[variant]\n    fingerprints = (\n        "split_sha256", "ground_truth_sha256", "checkpoint_sha256",\n        "clip_backbone_sha256", "frames_per_feature", "visual_block_length",\n        "feature_inventory_sha256" if prefix == "ucf" else "source_features_archive_sha256",\n    )\n    for key in fingerprints:\n        if control[key] != treatment[key]:\n            raise ValueError(f"Đầu vào {reference}/{variant} khác nhau: {key}")\n    id_key = "video_id" if prefix == "ucf" else "official_path"\n    branch = "classification" if prefix == "ucf" else "branch_2"\n    with (\n        np.load(paths[reference].with_name("predictions.npz"), allow_pickle=False) as old,\n        np.load(paths[variant].with_name("predictions.npz"), allow_pickle=False) as new,\n    ):\n        for key in (id_key, "video_label", "frame_gt", "snippet_offsets", "score_columns"):\n            if not np.array_equal(old[key], new[key]):\n                raise ValueError(f"Dự đoán {reference}/{variant} lệch {key}")\n        raw_gt = old["frame_gt"].reshape(-1)\n        offsets = old["snippet_offsets"]\n        labels = old["video_label"]\n        columns = old["score_columns"].tolist()\n        if branch not in columns or len(set(columns)) != len(columns):\n            raise ValueError("Thiếu nhánh chính hoặc cột điểm bị trùng.")\n        column = columns.index(branch)\n        old_scores = old["scores"][:, column]\n        new_scores = new["scores"][:, column]\n        video_count = len(old[id_key])\n    frames = control["frames_per_feature"]\n    width = control["visual_block_length"]\n    if (\n        not np.array_equal(np.unique(raw_gt), [0, 1])\n        or len(offsets) != video_count + 1 or offsets[0] != 0\n        or (np.diff(offsets) <= 0).any() or offsets[-1] * frames != len(raw_gt)\n        or len(old_scores) != offsets[-1] or len(new_scores) != offsets[-1]\n        or not np.isfinite(old_scores).all() or not np.isfinite(new_scores).all()\n    ):\n        raise ValueError("GT, số video, offset hoặc điểm không hợp lệ.")\n    gt = raw_gt.astype(bool)\n    frame_offsets = offsets * frames\n    old_frames, new_frames = (np.repeat(s, frames) for s in (old_scores, new_scores))\n    spans, owners = event_intervals(gt, frame_offsets)\n    abnormal_video = np.char.lower(labels.astype(str)) != ("normal" if prefix == "ucf" else "a")\n    abnormal_background = np.repeat(abnormal_video, np.diff(frame_offsets)) & ~gt\n    # XD upstream may export textual labels rather than original codes.\n    if prefix == "xd":\n        normal = np.isin(np.char.lower(labels.astype(str)), ["a", "normal"])\n        if not normal.any() or normal.all():\n            raise ValueError("Không nhận diện được nhãn video Normal của XD.")\n        abnormal_background = np.repeat(~normal, np.diff(frame_offsets)) & ~gt\n    if not len(spans):\n        raise ValueError("Không có khoảng bất thường để chấm độ bao phủ.")\n    near = np.zeros(offsets[-1], dtype=bool)\n    for start, stop in zip(offsets[:-1], offsets[1:], strict=True):\n        for edge in range(start + width, stop, width):\n            near[max(start, edge - 8):min(stop, edge + 8)] = True\n    near = np.repeat(near, frames)\n    metric = roc_auc_score if prefix == "ucf" else average_precision_score\n    key = "roc_auc" if prefix == "ucf" else "frame_ap"\n    for values, report in ((old_frames, control), (new_frames, treatment)):\n        if not math.isclose(metric(gt, values), report["metrics"][branch][key], abs_tol=1e-9):\n            raise ValueError("Metric nhánh chính từ predictions.npz lệch run.json.")\n    report_metrics = [report["metrics"][branch] for report in (control, treatment)]\n    benchmark_ci = {\n        metric_key: paired_benchmark_ci(\n            gt, (old_scores, new_scores), offsets, frames, metric_key,\n            [values[metric_key] for values in report_metrics], seed, bootstrap,\n        )\n        for metric_key in ("roc_auc", "ap" if prefix == "ucf" else "frame_ap")\n    }\n    if prefix == "ucf":\n        steps = np.diff(offsets)\n        keep = np.repeat(abnormal_video, steps)\n        sub_offsets = np.r_[0, np.cumsum(steps[abnormal_video])]\n        benchmark_ci["ano_auc"] = paired_benchmark_ci(\n            gt[np.repeat(keep, frames)], (old_scores[keep], new_scores[keep]),\n            sub_offsets, frames, "roc_auc", [values["ano_auc"] for values in report_metrics],\n            seed, bootstrap,\n        )\n    region_metrics = {}\n    for name, mask in (("near_internal_boundary", near), ("outside_boundary_region", ~near)):\n        labels_in_region = gt[mask]\n        positives = int(labels_in_region.sum())\n        values = [float(metric(labels_in_region, s[mask])) for s in (old_frames, new_frames)] \\\n            if 0 < positives < len(labels_in_region) else [None, None]\n        region_metrics[name] = {\n            "frames": int(mask.sum()), "positives": positives,\n            "reference": values[0], "treatment": values[1],\n            "treatment_minus_reference": values[1] - values[0] if values[0] is not None else None,\n        }\n    short = (spans[:, 1] - spans[:, 0]) <= 48\n    mid = ((spans[:, 1] - spans[:, 0]) > 48) & ((spans[:, 1] - spans[:, 0]) <= 160)\n    long = (spans[:, 1] - spans[:, 0]) > 160\n    within = [\n        [roc_auc_score(gt[start:stop], scores[start:stop]) for scores in (old_frames, new_frames)]\n        for start, stop in zip(frame_offsets[:-1], frame_offsets[1:], strict=True)\n        if gt[start:stop].any() and not gt[start:stop].all()\n    ]\n    within_scores = np.mean(within, axis=0).tolist() if within else [None, None]\n    curve = []\n    for fpr in fprs:\n        points = [\n            operating_point(gt, s, s[~gt], abnormal_background, spans, fpr)\n            for s in (old_frames, new_frames)\n        ]\n        (left, left_recall), (right, right_recall) = points\n        diff = right_recall - left_recall\n        for name, bucket in (("mid", mid), ("long", long)):\n            for point, recalls in ((left, left_recall), (right, right_recall)):\n                point[f"mean_{name}_event_frame_recall"] = (\n                    float(recalls[bucket].mean()) if bucket.any() else None\n                )\n        curve.append({\n            "background_fpr_cap": fpr, "reference": left, "treatment": right,\n            "treatment_minus_reference_event_recall": float(diff.mean()),\n            "paired_bootstrap": paired_event_ci(diff, owners, video_count, seed, bootstrap),\n            "short_event_paired_bootstrap": (\n                paired_event_ci(diff[short], owners[short], video_count, seed, bootstrap)\n                if short.any() else None\n            ),\n        })\n    peaks = [\n        float(new_frames[start:stop].max() - old_frames[start:stop].max())\n        for start, stop in spans[short]\n    ]\n    return {\n        "reference_variant": reference, "treatment_variant": variant,\n        "event_count": len(spans),\n        "short_event_count": int(short.sum()),\n        "mid_event_count": int(mid.sum()),\n        "long_event_count": int(long.sum()),\n        "within_video_roc_auc": {\n            "videos_with_both_frame_labels": len(within),\n            "reference_macro_auc": within_scores[0],\n            "treatment_macro_auc": within_scores[1],\n        },\n        "short_event_peak_lower_count": sum(value < 0 for value in peaks),\n        "short_event_mean_peak_delta": float(np.mean(peaks)) if peaks else None,\n        "benchmark_paired_bootstrap": benchmark_ci,\n        "region_metric": key, "region_metrics": region_metrics,\n        "matched_background_fpr": curve,\n    }\n\n\n'}
TRAINER_SOURCE = 'import csv\nimport hashlib\nimport importlib\nimport io\nimport json\nimport random\nimport subprocess\nimport sys\nimport tempfile\nimport time\nfrom pathlib import PurePosixPath\nfrom shutil import copyfile, copytree\nfrom zipfile import ZipFile\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom sklearn.metrics import average_precision_score\nfrom sklearn.model_selection import GroupShuffleSplit\n\nfrom head import TEMPORAL_LAGS, TemporalHead, anomaly_logits\nfrom v7_windows import overlap_forward\n\n\nCOMMIT = "c41067f07d252efcda18008bea367886070c33b0"\nCLIP_SHA = "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f"\nUCF_CLASSES = (\n    "Normal", "Abuse", "Arrest", "Arson", "Assault", "Burglary", "Explosion",\n    "Fighting", "RoadAccidents", "Robbery", "Shooting", "Shoplifting", "Stealing", "Vandalism",\n)\nXD_LABELS = {\n    "A": "normal", "B1": "fighting", "B2": "shooting", "B4": "riot",\n    "B5": "abuse", "B6": "car accident", "G": "explosion",\n}\n\n\ndef file_sha(path):\n    with path.open("rb") as handle:\n        return hashlib.file_digest(handle, "sha256").hexdigest()\n\n\ndef source_basename(path):\n    return PurePosixPath(str(path).replace(chr(92), "/")).name\n\n\ndef video_id(path):\n    stem, sep, crop = source_basename(path).rpartition("__")\n    if not stem or not sep or crop not in {f"{i}.npy" for i in range(10)}:\n        raise ValueError(f"Tên feature VadCLIP sai: {path}")\n    return stem.casefold()\n\n\ndef checkout():\n    if not SRC_REPO.exists():\n        SRC_REPO.parent.mkdir(parents=True, exist_ok=True)\n        with tempfile.TemporaryDirectory(dir=SRC_REPO.parent) as tmp:\n            clone = Path(tmp) / "source"\n            subprocess.run(["git", "clone", "https://github.com/nwpu-zxr/VadCLIP.git", str(clone)], check=True)\n            subprocess.run(["git", "-C", str(clone), "checkout", "--detach", COMMIT], check=True)\n            clone.rename(SRC_REPO)\n    actual = subprocess.check_output(["git", "-C", str(SRC_REPO), "rev-parse", "HEAD"], text=True).strip()\n    dirty = subprocess.check_output(\n        ["git", "-C", str(SRC_REPO), "status", "--porcelain", "--untracked-files=all"], text=True,\n    ).strip()\n    if actual != COMMIT or dirty:\n        raise RuntimeError("Cần checkout VadCLIP sạch tại commit đã khóa.")\n\n\ndef official_csv(default, repo):\n    path = repo / "list" / source_basename(default)\n    with path.open(newline="", encoding="utf-8-sig") as handle:\n        reader = csv.DictReader(handle)\n        if not reader.fieldnames or not {"path", "label"} <= set(reader.fieldnames):\n            raise ValueError(f"Thiếu cột path/label: {path.name}")\n        rows = list(reader)\n    return rows, file_sha(path)\n\n\nclass FeaturePack:\n    def __init__(self, root, prefix, expected_manifest):\n        name = f"vadclip_{prefix}_train_v5"\n        if root.is_file():\n            candidates = [root]\n        else:\n            candidates = sorted(root.rglob(f"{name}_manifest.json"))\n            if not candidates:\n                candidates = sorted(root.rglob(f"{name}.zip"))\n        if len(candidates) != 1:\n            raise ValueError(f"Cần đúng một gói {name}.zip hoặc manifest giải nén. Tìm thấy {len(candidates)}.")\n        location = candidates[0]\n        self.archive = None\n        self.directory = location.parent\n        manifest_name = f"{name}_manifest.json"\n        if location.suffix == ".zip":\n            self.archive = ZipFile(location)\n            payload = self.archive.read(manifest_name)\n        else:\n            payload = location.read_bytes()\n        self.manifest_sha256 = hashlib.sha256(payload).hexdigest()\n        if self.manifest_sha256 != expected_manifest:\n            raise ValueError(f"Sai hash manifest gói {name}.")\n        content = json.loads(payload)\n        crop, count = (5, 1610) if prefix == "ucf" else (0, 3954)\n        if (content["dataset"] != prefix or content["crop"] != crop\n                or content["schema"] != "vadclip_v5_train_v1" or len(content["features"]) != count):\n            raise ValueError(f"Sai schema hoặc số video trong gói {name}.")\n        self.features = {}\n        for idx, item in enumerate(content["features"], 1):\n            if item["path"] != f"features/{prefix}_{idx:05d}.npy":\n                raise ValueError("Tên vật lý trong manifest train không hợp lệ.")\n            key = video_id(item["original"])\n            if key in self.features:\n                raise ValueError(f"ID trùng trong gói train: {key}")\n            self.features[key] = item\n        self.checked = set()\n\n    def array(self, item):\n        path = item["path"]\n        data = self.archive.read(path) if self.archive else (self.directory / path).read_bytes()\n        if path not in self.checked:\n            if hashlib.sha256(data).hexdigest() != item["sha256"]:\n                raise ValueError(f"SHA-256 sai cho feature train: {path}")\n            self.checked.add(path)\n        array = np.load(io.BytesIO(data), allow_pickle=False)\n        if (array.shape != (item["steps"], 512) or not len(array)\n                or array.dtype.kind != "f" or not np.isfinite(array).all()):\n            raise ValueError(f"Shape/dtype/giá trị feature train sai: {path}")\n        return array\n\n    def close(self):\n        if self.archive is not None:\n            self.archive.close()\n\n\ndef select_train_rows(train_rows, features, encode_labels):\n    """Gộp các crop của cùng một video, ưu tiên crop có trong gói feature."""\n    selected = {}\n    for row in train_rows:\n        key = video_id(row["path"])\n        feature = features.get(key)\n        matches_pack_crop = (\n            feature is not None\n            and source_basename(row["path"]).casefold()\n            == source_basename(feature["original"]).casefold()\n        )\n        previous = selected.get(key)\n        if previous is not None:\n            previous_row, previous_matches = previous\n            if previous_row["label"] != row["label"]:\n                encoded = encode_labels([previous_row["label"], row["label"]])\n                if not torch.equal(encoded[0], encoded[1]):\n                    raise ValueError(\n                        f"Nhãn khác nhau giữa các crop của video {key}: "\n                        f"{previous_row[\'label\']} và {row[\'label\']}."\n                    )\n            if previous_matches or not matches_pack_crop:\n                continue\n        selected[key] = (row, matches_pack_crop)\n    return [row for row, _ in selected.values()]\n\n\ndef dataset_split(train_rows, test_rows, features, encode_labels, prefix, seed):\n    train_ids = [video_id(row["path"]) for row in train_rows]\n    test_ids = {video_id(row["path"]) for row in test_rows}\n    if set(train_ids) & test_ids:\n        raise ValueError("ID official train giao official test.")\n    if set(train_ids) != features.keys():\n        missing = set(train_ids) - features.keys()\n        extra = features.keys() - set(train_ids)\n        raise ValueError(f"Gói không khớp CSV train. Thiếu {len(missing)}, thừa {len(extra)}. Mẫu: {sorted(missing)[:4]}.")\n    labels = encode_labels([row["label"] for row in train_rows]).float().cpu()\n    if labels.ndim != 2 or (labels.sum(1) <= 0).any():\n        raise ValueError("Nhãn video train có hàng rỗng.")\n    abnormal = (labels[:, 0] == 0).numpy()\n    if (labels[:, 0].bool() & labels[:, 1:].sum(1).bool()).any():\n        raise ValueError("Nhãn Normal đồng thời có nhãn bất thường.")\n    if (labels.sum(0) <= 0).any():\n        raise ValueError("Official train thiếu lớp trong label map.")\n    groups = [vid.split("__#", 1)[0] if prefix == "xd" else vid for vid in train_ids]\n    test_groups = {vid.split("__#", 1)[0] if prefix == "xd" else vid for vid in test_ids}\n    excluded = np.flatnonzero(np.isin(groups, sorted(test_groups)))\n    eligible = np.flatnonzero(~np.isin(groups, sorted(test_groups)))\n    if len(eligible) < 2 or (labels[eligible].sum(0) <= 0).any():\n        raise ValueError("Lọc nhóm tên nguồn khiến train/dev thiếu dữ liệu hoặc thiếu lớp.")\n    eligible_groups = np.asarray(groups)[eligible]\n    candidates = GroupShuffleSplit(n_splits=30, test_size=0.10, random_state=seed)\n    for train_positions, dev_positions in candidates.split(eligible, abnormal[eligible], eligible_groups):\n        train, dev = eligible[train_positions], eligible[dev_positions]\n        if (len(np.unique(abnormal[train])) == 2 and len(np.unique(abnormal[dev])) == 2\n                and bool((labels[train].sum(0) > 0).all())):\n            assert not set(np.asarray(groups)[train]) & set(np.asarray(groups)[dev])\n            return train_ids, groups, labels, abnormal, train, dev, excluded, len(set(groups) & test_groups)\n    raise ValueError("Chưa tìm được train/development đủ hai nhóm và độ phủ train.")\n\n\ndef balanced_batches(indices, abnormal, half, seed):\n    rng = np.random.default_rng(seed)\n    positive = rng.permutation(indices[abnormal[indices]])\n    normal = rng.permutation(indices[~abnormal[indices]])\n    batches = (max(len(positive), len(normal)) + half - 1) // half\n    positive = np.resize(positive, batches * half).reshape(batches, half)\n    normal = np.resize(normal, batches * half).reshape(batches, half)\n    return np.concatenate((normal, positive), axis=1)\n\n\n\n\n\ndef seed_all(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n\n\ndef cache_v7(model, context, indices, make_mask):\n    pack, ids, labels, _, args, device, prompts, _ = context\n    cache = {}\n    width = args.visual_length\n    checked = 0\n    with torch.inference_mode():\n        for i in indices:\n            key = ids[i]\n            item = pack.features[key]\n            array = pack.array(item)\n            steps = len(array)\n            blocks = (steps + width - 1) // width\n            x = torch.zeros((blocks * width, 512), device=device)\n            x[:steps] = torch.as_tensor(array, dtype=torch.float32, device=device)\n            lengths = (steps - torch.arange(blocks) * width).clamp(max=width)\n            _, c, a = overlap_forward(model, x.reshape(blocks, width, 512),\n                                      make_mask(lengths, width).to(device), prompts, lengths,\n                                      make_mask)\n            c, binary_a = anomaly_logits(c, a)\n            allowed = labels[i, 1:].bool().to(device)\n            logits = a.float().reshape(-1, a.shape[-1])[:steps]\n            semantic = (logits[:, 1:][:, allowed].logsumexp(-1) - logits[:, 0]\n                        if allowed.any() else binary_a.reshape(-1)[:steps])\n            teacher = (c.reshape(-1)[:steps].sigmoid() + semantic.sigmoid()) / 2\n            neighbor = teacher if steps == 1 else torch.maximum(\n                F.pad(teacher[:-1], (1, 0)), F.pad(teacher[1:], (0, 1)))\n            base = (c if PREFIX == "ucf" else binary_a).reshape(-1)[:steps]\n            if not torch.isfinite(base).all() or not torch.isfinite(teacher).all():\n                raise ValueError(f"Invalid VadCLIP logits: {key}")\n            cache[int(i)] = (array.astype(np.float16), base.cpu().numpy(),\n                             torch.minimum(teacher, neighbor).cpu().numpy())\n            checked += 1\n            if checked % 100 == 0 or checked == len(indices):\n                print(f"{PREFIX}: cached V7 + CLIP {checked}/{len(indices)}", flush=True)\n    return cache, {"videos": checked, "temporal_lags": TEMPORAL_LAGS}\n\n\ndef candidates(teacher, positive, rng):\n    # Explicitly skip ambiguous positives and distinct isolated high peaks.\n    if positive:\n        valid = np.flatnonzero(teacher >= 0.50)\n        rank = valid[np.argsort(-teacher[valid], kind="stable")]\n        return rank[:min(16, max(1, len(teacher) // 16))] if len(teacher) > 1 else rank[:0]\n    return rng.choice(len(teacher), min(16, len(teacher)), replace=False)\n\n\ndef fit_one(context, schedule, dev, cache, variant):\n    _, _, _, abnormal, _, device, _, _ = context\n    seed_all(SEED)\n    head = TemporalHead().to(device)\n    use_temporal = variant != "clip_only"\n    opt = torch.optim.AdamW(head.parameters(), lr=0.001)\n    total_pairs = 0\n    history = []\n    for epoch, batches in enumerate(schedule):\n        head.train()\n        rng = np.random.default_rng(SEED + epoch)\n        pair_rng = np.random.default_rng(SEED + len(schedule) + epoch)\n        losses = []\n        for batch in batches:\n            scores, pool = [], {False: [], True: []}\n            for i in batch:\n                clip, base, teacher = cache[int(i)]\n                x = torch.as_tensor(clip, dtype=torch.float32, device=device)\n                b = torch.as_tensor(base, device=device)\n                signal = head(x, use_temporal)\n                z = b + signal\n                scores.append(z.topk(1 + len(z) // 16).values.mean())\n                pos = bool(abnormal[i])\n                ix = candidates(teacher, pos, rng)\n                if len(ix):\n                    ix = torch.as_tensor(ix, device=device)\n                    pool[pos].append((x[ix].detach(), z[ix]))\n            labels = torch.as_tensor(abnormal[batch], device=device, dtype=torch.float32)\n            loss = F.binary_cross_entropy_with_logits(torch.stack(scores), labels)\n            if pool[True] and pool[False]:\n                px, pz = (torch.cat(items) for items in zip(*pool[True], strict=True))\n                nx, nz = (torch.cat(items) for items in zip(*pool[False], strict=True))\n                if variant == "v18":\n                    match = (F.normalize(px, dim=-1) @ F.normalize(nx, dim=-1).T).argmax(1)\n                else:\n                    match = torch.as_tensor(pair_rng.integers(len(nz), size=len(pz)), device=device)\n                loss = loss + 0.5 * F.softplus(0.5 - pz + nz[match]).mean()\n                total_pairs += len(pz)\n            if not torch.isfinite(loss):\n                raise ValueError(f"Invalid V18 training loss: {variant}")\n            opt.zero_grad(set_to_none=True)\n            loss.backward()\n            opt.step()\n            losses.append(float(loss.detach()))\n        history.append(float(np.mean(losses)))\n        print(f"{PREFIX}/{variant}: epoch {epoch+1}, loss={history[-1]:.5f}", flush=True)\n    head.eval()\n    strengths = (0., 0.25, 0.5, 1.)\n    dev_scores = {strength: [] for strength in strengths}\n    with torch.no_grad():\n        for i in dev:\n            clip, base, _ = cache[int(i)]\n            x = torch.as_tensor(clip, dtype=torch.float32, device=device)\n            delta = head(x, use_temporal)\n            b = torch.as_tensor(base, device=device)\n            for strength in strengths:\n                dev_scores[strength].append(float((b + strength * delta).topk(1 + len(b)//16).values.mean()))\n    curve = {s: float(average_precision_score(abnormal[dev], scores))\n             for s, scores in dev_scores.items()}\n    strength = max(strengths, key=curve.get)\n    path = OUTPUT / f"{variant}.pth"\n    save_checkpoint(head, path)\n    return {"strength": strength, "dev_video_ap": curve, "paired_examples": total_pairs,\n            "epoch_loss": history, "checkpoint_file": path.name, "checkpoint_sha256": file_sha(path)}\n\n\ndef fit_variants(model, context, train, dev, make_mask):\n    _, _, _, abnormal, _, _, _, _ = context\n    indices = np.unique(np.r_[train, dev])\n    schedule = [balanced_batches(train, abnormal, 2 if SMOKE_ONLY else HALF_BATCH, SEED + epoch)\n                for epoch in range(1 if SMOKE_ONLY else TEMPORAL_CONFIG["epochs"])]\n    if SMOKE_ONLY:\n        schedule = [schedule[0][:2]]\n        indices = np.unique(np.r_[schedule[0].reshape(-1), dev])\n    cache, receipt = cache_v7(model, context, indices, make_mask)\n    result = {variant: fit_one(context, schedule, dev, cache, variant) for variant in VARIANTS}\n    if not SMOKE_ONLY and any(r["paired_examples"] == 0 for r in result.values()):\n        raise ValueError("No valid positive-negative pairs. Inspect training candidate coverage.")\n    return result, receipt\n\n\ndef save_checkpoint(model, path):\n    pending = path.with_suffix(".pth.partial")\n    torch.save(model.state_dict(), pending)\n    pending.replace(path)\n\n\ndef run_training():\n    OUTPUT.mkdir(parents=True, exist_ok=True)\n    if not torch.cuda.is_available():\n        raise RuntimeError("V18 cần Accelerator GPU trong Kaggle.")\n    checkout()\n    if CLIP_WEIGHTS is not None:\n        source = Path(CLIP_WEIGHTS)\n        if file_sha(source) != CLIP_SHA:\n            raise ValueError("CLIP_WEIGHTS sai SHA-256 của OpenAI ViT-B-16.pt.")\n        cache = Path.home() / ".cache" / "clip"\n        cache.mkdir(parents=True, exist_ok=True)\n        target = cache / "ViT-B-16.pt"\n        if source.resolve() != target.resolve() and (\n            not target.is_file() or file_sha(target) != CLIP_SHA\n        ):\n            with tempfile.TemporaryDirectory(dir=cache) as cache_tmp:\n                pending = Path(cache_tmp) / target.name\n                copyfile(source, pending)\n                if file_sha(pending) != CLIP_SHA:\n                    raise ValueError("Bản sao CLIP sai SHA-256.")\n                pending.replace(target)\n    packages = {"ftfy": "ftfy", "regex": "regex", "tqdm": "tqdm", "pandas": "pandas"}\n    missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]\n    if missing:\n        subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)\n    device = torch.device("cuda:0")\n    torch.cuda.set_device(device)\n    with tempfile.TemporaryDirectory(prefix="vadclip_v18_source_") as work:\n        runtime = Path(work) / "src"\n        copytree(SRC_REPO / "src", runtime)\n        clip_file = runtime / "clip" / "clip.py"\n        text = clip_file.read_text(encoding="utf-8")\n        old = "from pkg_resources import packaging"\n        if text.count(old) != 1:\n            raise ValueError("Mã CLIP upstream khác commit đã khóa.")\n        clip_file.write_text(text.replace(old, "import packaging.version"), encoding="utf-8")\n        sys.path.insert(0, str(runtime))\n        from model import CLIPVAD\n        from utils.tools import get_batch_label, get_batch_mask, get_prompt_text\n\n        args = importlib.import_module(f"{PREFIX}_option").parser.parse_args([])\n        train_rows, train_sha = official_csv(args.train_list, SRC_REPO)\n        test_rows, test_sha = official_csv(args.test_list, SRC_REPO)\n        label_map = {name: name.lower() for name in UCF_CLASSES} if PREFIX == "ucf" else XD_LABELS\n        if PREFIX == "ucf":\n            label_map["RoadAccidents"] = "roadAccidents"\n        prompts = get_prompt_text(label_map)\n        if len(prompts) != args.classes_num or prompts[0].casefold() != "normal":\n            raise ValueError(f"Thứ tự lớp VadCLIP sai: {prompts}")\n        primary = "C" if PREFIX == "ucf" else "A"\n\n        pack = FeaturePack(Path(TRAIN_ROOT) if TRAIN_ROOT else INPUT, PREFIX, MANIFEST_SHA256)\n        try:\n            def encode_labels(labels):\n                return get_batch_label(labels, prompts, label_map)\n\n            train_csv_rows = len(train_rows)\n            train_rows = select_train_rows(train_rows, pack.features, encode_labels)\n            print(\n                f"{PREFIX}: CSV train có {train_csv_rows:,} dòng, "\n                f"gộp thành {len(train_rows):,} video theo gói feature.",\n                flush=True,\n            )\n            split = dataset_split(train_rows, test_rows, pack.features, encode_labels, PREFIX, SEED)\n            train_ids, groups, labels, abnormal, train, dev, excluded, source_overlap = split\n            if labels.shape != (len(train_rows), len(prompts)):\n                raise ValueError("Kích thước ma trận nhãn train sai.")\n            manifest = [\n                {"id": train_ids[i], "group": groups[i], "label": train_rows[i]["label"],\n                 "split": name, "sha256": pack.features[train_ids[i]]["sha256"]}\n                for name, indices in (("train", train), ("development", dev), ("excluded_source_name", excluded))\n                for i in indices\n            ]\n            manifest_path = OUTPUT / "split_manifest.json"\n            manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n            print(f"{PREFIX}: train={len(train)}, development={len(dev)}, "\n                  f"official test={len(test_rows)}, nhóm tên nguồn trùng trước lọc={source_overlap}, "\n                  f"clip loại={len(excluded)}", flush=True)\n            if SMOKE_ONLY:\n                dev = np.r_[dev[~abnormal[dev]][:2], dev[abnormal[dev]][:2]]\n            seed_all(SEED)\n            model = CLIPVAD(\n                args.classes_num, args.embed_dim, args.visual_length, args.visual_width,\n                args.visual_head, args.visual_layers, args.attn_window,\n                args.prompt_prefix, args.prompt_postfix, device,\n            ).to(device)\n            if file_sha(V0_PATH) != V0_SHA256:\n                raise ValueError("Frozen V0 checkpoint SHA-256 mismatch.")\n            state = torch.load(V0_PATH, map_location="cpu", weights_only=True)\n            if isinstance(state, dict):\n                for field in ("state_dict", "model_state_dict"):\n                    if field in state:\n                        state = state[field]\n                        break\n            model.load_state_dict(state, strict=True)\n            del state\n            model.requires_grad_(False).eval()\n            context = (pack, train_ids, labels, abnormal, args, device, prompts, primary)\n            histories, logit_cache = fit_variants(model, context, train, dev, get_batch_mask)\n\n        finally:\n            pack.close()\n    report = {\n        "mode": "smoke" if SMOKE_ONLY else "full",\n        "prefix": PREFIX, "commit": COMMIT, "seed": SEED, "primary_branch": primary,\n        "max_epoch": 1 if SMOKE_ONLY else TEMPORAL_CONFIG["epochs"],\n        "temporal_config": TEMPORAL_CONFIG, "frozen_v0_cache": logit_cache, "frozen_v0_sha256": V0_SHA256,\n        "half_batch": 2 if SMOKE_ONLY else HALF_BATCH,\n        "feature_manifest_sha256": MANIFEST_SHA256, "train_csv_sha256": train_sha,\n        "test_csv_sha256": test_sha, "split_manifest_sha256": file_sha(manifest_path),\n        "train_csv_row_count": train_csv_rows,\n        "train_video_count": len(train_rows),\n        "train_count": len(train), "development_count": len(dev),\n        "source_name_groups_in_both_splits_before_filter": source_overlap,\n        "excluded_source_name_clip_count": len(excluded),\n        "verified_feature_count": len(pack.checked), "variants": histories,\n    }\n    if not SMOKE_ONLY and len(pack.checked) != len(train_rows) - len(excluded):\n        raise RuntimeError("Chưa đọc và kiểm hash toàn bộ feature train/development.")\n    path = OUTPUT / "training.json"\n    pending = path.with_suffix(".json.partial")\n    pending.write_text(json.dumps(report, indent=2), encoding="utf-8")\n    pending.replace(path)\n\n\nif __name__ == "__main__":\n    run_training()\n'
UCF_RUNNER = '\nCOMMIT = "c41067f07d252efcda18008bea367886070c33b0"\nUPSTREAM_URL = "https://github.com/nwpu-zxr/VadCLIP.git"\nEXPECTED_INPUTS = {\n    "checkpoint_sha256": "d064aba05190ce5f6652151e6259b24ca6dda83a1e6289bf343afb6f905a2c79",\n    "split_sha256": "d621f4ccfb3e26274d215ad53b83fd1f596bb7e9f85cde69e01399db3da97c7f",\n    "ground_truth_sha256": "c6448801e0f73d19adf82506cd58d60a90c4f0913e49dcc1ad1d677ee12d6301",\n    "annotation_sha256": "82d581cd4685480ffda23eb0fe1bca441b672bfe99c457adca2df4cf4780c503",\n    "feature_inventory_sha256": "8aa3a3c58f9bbbcc8082e024f3892e014f059374e8b52ca737ae709945210026"\n}\n\nimport csv\nimport hashlib\nimport importlib.util\nimport json\nimport os\nimport re\nimport subprocess\nimport sys\nimport tempfile\nimport time\nfrom collections import defaultdict\nfrom shutil import copyfile, copyfileobj, copytree\nfrom urllib.error import URLError\nfrom zipfile import BadZipFile, ZipFile\n\nif not INPUT.is_dir():\n    raise FileNotFoundError("Import notebook trên Kaggle và gắn input trước.")\n\ndef git(*args):\n    return subprocess.check_output(["git", *map(str, args)], text=True).strip()\n\n\nif not SRC_REPO.exists():\n    SRC_REPO.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.TemporaryDirectory(dir=SRC_REPO.parent) as checkout_dir:\n        checkout = Path(checkout_dir) / "source"\n        subprocess.run(["git", "clone", UPSTREAM_URL, str(checkout)], check=True)\n        subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", COMMIT], check=True)\n        checkout.rename(SRC_REPO)\nif git("-C", SRC_REPO, "rev-parse", "HEAD") != COMMIT:\n    raise RuntimeError(f"Mã nguồn tại {SRC_REPO} sai commit. Hãy dùng thư mục nguồn mới.")\nif git("-C", SRC_REPO, "status", "--porcelain", "--untracked-files=no"):\n    raise RuntimeError(f"Mã nguồn tại {SRC_REPO} có thay đổi. Hãy dùng checkout sạch.")\n\npackages = {\n    "ftfy": "ftfy", "packaging": "packaging", "regex": "regex",\n    "tqdm": "tqdm", "sklearn": "scikit-learn",\n}\nmissing_packages = [pkg for mod, pkg in packages.items() if importlib.util.find_spec(mod) is None]\nif missing_packages:\n    subprocess.run([sys.executable, "-m", "pip", "install", *missing_packages], check=True)\n\nnp = importlib.import_module("numpy")\n\n# Modern setuptools no longer ships pkg_resources. Adapt the import in a copy.\nstage_directory = tempfile.TemporaryDirectory(prefix="vadclip_ucf_v5_")\nstaging = Path(stage_directory.name)\nruntime_src = staging / "runtime"\ncopytree(SRC_REPO / "src", runtime_src, dirs_exist_ok=True)\nfrom v7_windows import overlap_forward\n\nclip_py = runtime_src / "clip" / "clip.py"\nclip_source = clip_py.read_text(encoding="utf-8")\nold_import = "from pkg_resources import packaging"\nif clip_source.count(old_import) != 1:\n    raise RuntimeError("Không tìm thấy import CLIP cần sửa tương thích. Kiểm commit.")\nclip_py.write_text(clip_source.replace(old_import, "import packaging.version"), encoding="utf-8")\n\nOUTPUT.mkdir(parents=True, exist_ok=True)\nprint("Commit VadCLIP", COMMIT)\nprint(f"Chấm UCF-Crime: {VARIANT}")\n\nCLASSES = (\n    "Normal", "Abuse", "Arrest", "Arson", "Assault", "Burglary", "Explosion",\n    "Fighting", "RoadAccidents", "Robbery", "Shooting", "Shoplifting", "Stealing", "Vandalism",\n)\nFRAMES_PER_FEATURE = 16\nFEATURE_DIM = 512\nVIDEO_COUNT = 290\nCLIP_SHA256 = "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f"\n\n\ndef sha256(path):\n    with path.open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef read_split(list_root):\n    """Đọc theo đúng thứ tự repo. Tính khoảng [start, end) theo frame gốc."""\n    with (list_root / "ucf_CLIP_rgbtest.csv").open(newline="", encoding="utf-8") as stream:\n        entries = list(csv.DictReader(stream))\n    annotations = {}\n    for line in (list_root / "Temporal_Anomaly_Annotation.txt").read_text().splitlines():\n        if not line.strip():\n            continue\n        name, label, *bounds = line.split()\n        video_id = Path(name).stem\n        if len(bounds) != 4 or video_id in annotations:\n            raise ValueError(f"Annotation không hợp lệ: {video_id}")\n        intervals = []\n        for start, end in zip(bounds[::2], bounds[1::2]):\n            lo, hi = int(start), int(end)\n            if (lo, hi) == (-1, -1):\n                continue\n            if not 0 <= lo < hi:\n                raise ValueError(f"Khoảng frame không hợp lệ: {video_id} {(lo, hi)}")\n            intervals.append((lo, hi))\n        annotations[video_id] = (label, intervals)\n\n    rows = []\n    for entry in entries:\n        feature_name = Path(entry["path"]).name\n        match = re.fullmatch(r"(.+)__([0-9])\\.npy", feature_name)\n        if match is None:\n            raise ValueError(f"Tên feature sai: {feature_name}")\n        video_id, crop = match.groups()\n        label, intervals = annotations[video_id]\n        if label != entry["label"] or label not in CLASSES:\n            raise ValueError(f"Nhãn không khớp giữa CSV và annotation: {video_id}")\n        rows.append({\n            "id": video_id, "label": label, "feature_name": feature_name,\n            "crop": int(crop), "intervals": intervals,\n        })\n    if len(rows) != VIDEO_COUNT or len({row["id"] for row in rows}) != VIDEO_COUNT:\n        raise ValueError("Cần đúng 290 ID duy nhất từ split UCF-Crime chính thức.")\n    return rows\n\n\n\ndef stage_member(archive, member, destination):\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    with archive.open(member) as source, destination.open("wb") as target:\n        copyfileobj(source, target, length=1024 * 1024)\n    return destination\n\n\ndef find_inputs(root, keys):\n    """Tìm theo tên file chính thức, kể cả khi Kaggle giữ ZIP nguyên gói."""\n    found = defaultdict(list)\n    if root.is_file():\n        archive_path = root\n    elif root.is_dir():\n        for folder, _, filenames in os.walk(root):\n            for name in filenames:\n                key = name.casefold()\n                if key in keys:\n                    found[key].append(Path(folder) / name)\n        if found:\n            return found\n        archives = sorted(root.rglob("ucfclipfeatures_test290.zip"))\n        if len(archives) != 1:\n            raise RuntimeError(\n                "Gắn Dataset feature UCF-Crime của V0 hoặc chỉ định DATA_ROOT."\n            )\n        archive_path = archives[0]\n    else:\n        raise FileNotFoundError(f"Không thấy DATA_ROOT: {root}")\n\n    with ZipFile(archive_path) as archive:\n        for index, member in enumerate(archive.infolist()):\n            name = Path(member.filename).name\n            key = name.casefold()\n            if not member.is_dir() and key in keys:\n                destination = staging / "features" / str(index) / name\n                found[key].append(stage_member(archive, member, destination))\n    return found\n\n\ndef checkpoint_candidates():\n    if CHECKPOINT is not None:\n        path = Path(CHECKPOINT)\n        if not path.is_file():\n            raise FileNotFoundError(f"UCF_CHECKPOINT không trỏ đến file đang gắn: {path}")\n        return [path]\n\n    candidates = []\n    for path in INPUT.rglob("*"):\n        if path.suffix.lower() not in {".pth", ".zip"}:\n            continue\n        name = path.stem.casefold()\n        is_ucf_checkpoint = name.startswith("model_ucf") or ("ucf" in name and "checkpoint" in name)\n        if is_ucf_checkpoint and path.is_file():\n            candidates.append(path)\n    return sorted(candidates, key=lambda path: (path.suffix.lower() == ".zip", str(path)))\n\n\ndef choose_checkpoint():\n    expected = EXPECTED_INPUTS["checkpoint_sha256"]\n    candidates = checkpoint_candidates()\n    if not candidates:\n        raise FileNotFoundError(\n            "Không thấy checkpoint UCF trong /kaggle/input. Vào Add Input và gắn Dataset có "\n            "model_ucf.pth hoặc vadclip_ucf_checkpoint_upload.zip. Nếu tên khác, "\n            "đặt UCF_CHECKPOINT thành đường dẫn đầy đủ của file."\n        )\n\n    rejected = []\n    for path in candidates:\n        try:\n            if path.suffix.lower() == ".zip":\n                with ZipFile(path) as archive:\n                    members = [\n                        item for item in archive.infolist()\n                        if not item.is_dir() and Path(item.filename).name.casefold() == "model_ucf.pth"\n                    ]\n                    for member in members:\n                        with archive.open(member) as stream:\n                            digest = hashlib.file_digest(stream, "sha256").hexdigest()\n                        if digest == expected:\n                            print(f"Checkpoint UCF V0 khớp SHA-256: {path} ({member.filename})")\n                            return stage_member(archive, member, staging / "checkpoint" / "model_ucf.pth")\n                rejected.append(f"{path.name} không chứa model_ucf.pth khớp SHA-256")\n            elif sha256(path) == expected:\n                print(f"Checkpoint UCF V0 khớp SHA-256: {path}")\n                return path\n            else:\n                rejected.append(f"{path.name} sai SHA-256")\n        except (BadZipFile, OSError) as exc:\n            rejected.append(f"{path.name}: {exc}")\n    raise ValueError("Không tìm được checkpoint UCF đúng hash V0. " + "; ".join(rejected))\n\n\ndef input_fingerprint(rows, list_root, checkpoint_path):\n    inventory = [[row["id"], row["label"], row["steps"], row["feature_sha256"]] for row in rows]\n    inventory_bytes = json.dumps(inventory, separators=(",", ":")).encode()\n    return {\n        "checkpoint_sha256": sha256(checkpoint_path),\n        "split_sha256": sha256(list_root / "ucf_CLIP_rgbtest.csv"),\n        "ground_truth_sha256": sha256(list_root / "gt_ucf.npy"),\n        "annotation_sha256": sha256(list_root / "Temporal_Anomaly_Annotation.txt"),\n        "feature_inventory_sha256": hashlib.sha256(inventory_bytes).hexdigest(),\n    }\n\n\ndef prepare_clip_cache(source, cache_dir):\n    """Đặt file OpenAI đã xác minh vào đúng cache mà loader gốc đọc."""\n    if not source.is_file():\n        raise FileNotFoundError(f"Không thấy CLIP_WEIGHTS: {source}")\n    if sha256(source) != CLIP_SHA256:\n        raise ValueError("CLIP_WEIGHTS sai SHA-256 của OpenAI ViT-B-16.pt.")\n    target = cache_dir / "ViT-B-16.pt"\n    if target.is_file() and sha256(target) == CLIP_SHA256:\n        return target\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    temporary = cache_dir / "ViT-B-16.pt.vadclip-part"\n    copyfile(source, temporary)\n    if sha256(temporary) != CLIP_SHA256:\n        raise ValueError("Bản sao CLIP lỗi. Kiểm dung lượng và file input.")\n    temporary.replace(target)\n    return target\n\n\ndef feature_array(path):\n    array = np.load(path, mmap_mode="r", allow_pickle=False)\n    if array.ndim != 2 or array.shape[1] != FEATURE_DIM or array.shape[0] < 1:\n        raise ValueError(f"Feature cần dạng [T, 512], nhận {array.shape}: {path}")\n    if array.dtype not in (np.float16, np.float32, np.float64):\n        raise ValueError(f"Feature phải là số thực, nhận {array.dtype}: {path}")\n    return array\n\n\ndef frame_labels(row, length):\n    labels = np.zeros(length, dtype=np.uint8)\n    for start, end in row["intervals"]:\n        labels[start:end] = 1\n    return labels\n\n\ndef validate_ground_truth(rows, gt):\n    """Kiểm cả nội dung và thứ tự, ngoài phép kiểm tổng số frame."""\n    offset = 0\n    for row in rows:\n        stop = offset + row["steps"] * FRAMES_PER_FEATURE\n        expected = frame_labels(row, stop - offset)\n        if not np.array_equal(gt[offset:stop], expected):\n            raise ValueError(\n                f"Ground truth lệch tại {row[\'id\']} (frame offset {offset}, "\n                f"{row[\'steps\']} bước). Kiểm lại bản video/feature và frame count."\n            )\n        offset = stop\n    if offset != len(gt):\n        raise ValueError(f"Số frame không khớp: input={offset}, ground truth={len(gt)}")\n\n# Kiểm đầu vào và đối chiếu hash với V0 đã chạy.\nlist_root = SRC_REPO / "list"\nrows = read_split(list_root)\ndata_root = Path(DATA_ROOT) if DATA_ROOT is not None else INPUT\nexpected = {row["feature_name"].casefold(): row["feature_name"] for row in rows}\nfound = find_inputs(data_root, expected)\nmissing = sorted(expected[key] for key in expected if not found[key])\nambiguous = {\n    expected[key]: [str(path) for path in sorted(found[key])]\n    for key in expected if len(found[key]) > 1\n}\nif missing or ambiguous:\n    issue_path = OUTPUT / "input_issues.json"\n    issue_path.write_text(json.dumps({\n        "missing": missing,\n        "ambiguous": ambiguous,\n    }, indent=2), encoding="utf-8")\n    raise RuntimeError(\n        f"Thiếu {len(missing)} file, trùng {len(ambiguous)} ID. "\n        f"Xem {issue_path} và kiểm DATA_ROOT."\n    )\nfor row in rows:\n    path = found[row["feature_name"].casefold()][0]\n    row["path"] = path\n    row["steps"] = len(feature_array(path))\n    row["feature_sha256"] = sha256(path)\n\ngt = np.load(list_root / "gt_ucf.npy", allow_pickle=False)\nif gt.ndim != 1 or set(np.unique(gt)) != {0.0, 1.0}:\n    raise ValueError("Ground truth UCF cần vector nhị phân cấp frame.")\nvalidate_ground_truth(rows, gt)\ncheckpoint_path = choose_checkpoint()\ninput_hashes = input_fingerprint(rows, list_root, checkpoint_path)\nmismatches = [key for key, value in EXPECTED_INPUTS.items() if input_hashes[key] != value]\nif mismatches:\n    raise ValueError("Đầu vào khác lượt V0 đã đối chiếu: " + ", ".join(mismatches))\nprint(f"Đã khớp {len(rows)} video, {len(gt):,} frame và toàn bộ hash đầu vào với V0.")\n\nimport torch\n\nsys.path.insert(0, str(runtime_src))\nfrom model import CLIPVAD\nfrom ucf_option import parser as ucf_parser\nfrom utils.tools import get_batch_mask\n\nif not torch.cuda.is_available():\n    raise RuntimeError("Hãy bật GPU trong Kaggle. Mã VadCLIP gốc cần CUDA khi chạy forward.")\ndevice = torch.device("cuda:0")\ntorch.cuda.set_device(device)\ntorch.manual_seed(234)\n\n# Dùng đúng cấu hình UCF từ mã tác giả, không đọc cờ của kernel Jupyter.\nargs = ucf_parser.parse_args([])\nevaluated_checkpoint = checkpoint_path\nevaluated_sha256 = sha256(evaluated_checkpoint)\nstate = torch.load(evaluated_checkpoint, map_location="cpu", weights_only=True)\nif isinstance(state, dict):\n    for field in ("state_dict", "model_state_dict"):\n        if field in state:\n            state = state[field]\n            break\nif CLIP_WEIGHTS is not None:\n    prepare_clip_cache(Path(CLIP_WEIGHTS), Path.home() / ".cache" / "clip")\ntry:\n    model = CLIPVAD(\n        args.classes_num, args.embed_dim, args.visual_length, args.visual_width, args.visual_head,\n        args.visual_layers, args.attn_window, args.prompt_prefix, args.prompt_postfix, device,\n    )\nexcept URLError as exc:\n    raise RuntimeError(\n        "Không tải được CLIP ViT-B/16. Bật Internet hoặc gắn ViT-B-16.pt chính thức "\n        "và đặt CLIP_WEIGHTS bằng đường dẫn file."\n    ) from exc\ntry:\n    model.load_state_dict(state, strict=True)\nexcept (RuntimeError, TypeError) as exc:\n    raise RuntimeError(\n        f"Checkpoint không khớp model VadCLIP UCF-Crime: {evaluated_checkpoint}. "\n        "Kiểm lại file từ cột Model dòng UCF-Crime, khác file ViT-B-16.pt."\n    ) from exc\ndel state\nmodel.to(device).eval()\nprint("Đã nạp đầy đủ checkpoint trên", torch.cuda.get_device_name(device))\n\ndef predict_video(feature, model, device, block_length):\n    """Trả điểm [T, 2], lần lượt sigmoid(C) và 1-softmax(A)[Normal]."""\n    steps = len(feature)\n    if steps < 1 or not np.isfinite(feature).all():\n        raise ValueError("Feature rỗng hoặc chứa NaN/Inf.")\n    blocks = (steps + block_length - 1) // block_length\n    padded = np.zeros((blocks * block_length, FEATURE_DIM), dtype=np.float32)\n    padded[:steps] = feature\n    lengths = torch.tensor([\n        min(block_length, steps - i * block_length) for i in range(blocks)\n    ])\n    visual = torch.from_numpy(padded.reshape(blocks, block_length, FEATURE_DIM)).to(device)\n    mask = get_batch_mask(lengths, block_length).to(device)\n    with torch.no_grad():\n        if VARIANT != "v0":\n            _, logits_c, logits_a = overlap_forward(\n                model, visual, mask, list(CLASSES), lengths, get_batch_mask,\n            )\n        else:\n            _, logits_c, logits_a = model(visual, mask, list(CLASSES), lengths)\n        logits_c, logits_a = apply_head(\n            bundle, visual, lengths, logits_c, logits_a, "C")\n        c_score = logits_c.reshape(-1)[:steps].sigmoid()\n        a_score = 1 - logits_a.reshape(-1, len(CLASSES))[:steps].softmax(dim=-1)[:, 0]\n        scores = torch.stack((c_score, a_score), dim=1).cpu().numpy()\n    if scores.shape != (steps, 2) or not np.isfinite(scores).all():\n        raise ValueError(f"Điểm bất thường không hợp lệ: {scores.shape}")\n    return scores\n\n\ndef run_video(row):\n    feature = feature_array(row["path"])\n    if feature.shape != (row["steps"], FEATURE_DIM):\n        raise ValueError(f"Feature {row[\'id\']} sai shape: {feature.shape}")\n    return predict_video(feature, model, device, args.visual_length)\n\n\nfrom head import apply_head, load_head\n\nbundle = load_head(HEAD_PATH, STRENGTH, USE_TEMPORAL, device) if HEAD_PATH else None\n\n# Chọn theo thứ tự split và độ dài, trước khi xem điểm.\npilot = sorted({\n    next(i for i, row in enumerate(rows) if row["label"] != "Normal"),\n    next(i for i, row in enumerate(rows) if row["label"] == "Normal"),\n    max(range(len(rows)), key=lambda i: rows[i]["steps"]),\n})\ntorch.cuda.reset_peak_memory_stats(device)\nstarted = time.perf_counter()\nscores_by_index = {}\nfor i in pilot:\n    before = time.perf_counter()\n    scores_by_index[i] = run_video(rows[i])\n    print(rows[i]["id"], scores_by_index[i].shape, f"{time.perf_counter() - before:.2f}s")\nsmoke_seconds = time.perf_counter() - started\nprint(f"Smoke test hoàn tất trong {smoke_seconds:.2f}s.")\n\nif not SMOKE_ONLY:\n    for i, row in enumerate(rows):\n        if i not in scores_by_index:\n            scores_by_index[i] = run_video(row)\n        if (i + 1) % 25 == 0 or i + 1 == len(rows):\n            print(f"Đã chạy {i + 1}/{len(rows)} video", flush=True)\n\nfrom importlib.metadata import version\n\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\nSCORE_COLUMNS = ("classification", "alignment")\n\n\n\n\ndef score_benchmark(rows, snippet_scores, gt):\n    steps_per_video = np.asarray([row["steps"] for row in rows])\n    if snippet_scores.shape != (sum(steps_per_video), len(SCORE_COLUMNS)):\n        raise ValueError("Số điểm snippet hoặc cột không khớp.")\n    frame_scores = np.repeat(snippet_scores, FRAMES_PER_FEATURE, axis=0)\n    if len(frame_scores) != len(gt):\n        raise ValueError("Số điểm cấp frame không khớp ground truth.")\n    frame_mask = np.repeat(\n        [row["label"] != "Normal" for row in rows],\n        steps_per_video * FRAMES_PER_FEATURE,\n    )\n    metrics = {}\n    for column, name in enumerate(SCORE_COLUMNS):\n        scores = frame_scores[:, column]\n        metrics[name] = {\n            "roc_auc": float(roc_auc_score(gt, scores)),\n            "ap": float(average_precision_score(gt, scores)),\n            "ano_auc": float(roc_auc_score(gt[frame_mask], scores[frame_mask])),\n        }\n    return metrics\n\n\nselected = sorted(scores_by_index)\nif not SMOKE_ONLY and selected != list(range(len(rows))):\n    raise ValueError("Thiếu dự đoán. Cần chạy toàn bộ split trước khi chấm.")\nselected_rows = [rows[i] for i in selected]\nordered = [scores_by_index[i] for i in selected]\nif any(score.shape != (row["steps"], 2)\n       for row, score in zip(selected_rows, ordered, strict=True)):\n    raise ValueError("Số điểm từng video không khớp.")\nsteps_per_video = [row["steps"] for row in selected_rows]\nscores = np.concatenate(ordered, axis=0)\n\noutput_dir = OUTPUT / ("smoke" if SMOKE_ONLY else "full")\noutput_dir.mkdir(parents=True, exist_ok=True)\nmetadata = {\n    "run_kind": "smoke_inference" if SMOKE_ONLY else "full_checkpoint_inference",\n    "benchmark_scope": "ucf_crime_official_test",\n    "evaluation_status": "exploratory_on_previously_seen_official_test",\n    "commit": COMMIT,\n    "runtime_patch": "CLIP imports packaging.version directly",\n    **input_hashes,\n    "clip_backbone_sha256": CLIP_SHA256,\n    "frames_per_feature": FRAMES_PER_FEATURE,\n    "visual_block_length": args.visual_length,\n    "label_order": CLASSES,\n    "score_columns": SCORE_COLUMNS,\n    "evaluated_checkpoint_sha256": evaluated_sha256,\n    "python": sys.version.split()[0],\n    "libraries": {name: version(name) for name in (\n        "torch", "torchvision", "numpy", "scipy", "scikit-learn", "pillow",\n    )},\n    "cuda": torch.version.cuda,\n    "gpu": torch.cuda.get_device_name(device),\n    "smoke_seconds": smoke_seconds,\n    "inference_seconds": time.perf_counter() - started,\n    "peak_torch_vram_mib": torch.cuda.max_memory_allocated(device) / (1024 ** 2),\n    "video_count": len(selected_rows),\n    "videos": [{key: value for key, value in row.items() if key != "path"}\n               for row in selected_rows],\n    "metrics": None,\n}\nif not SMOKE_ONLY:\n    metrics = score_benchmark(rows, scores, gt)\n    metadata["metrics"] = metrics\n    print(json.dumps(metrics, indent=2))\n\nnp.savez_compressed(\n    output_dir / "predictions.npz",\n    video_id=np.asarray([row["id"] for row in selected_rows]),\n    video_label=np.asarray([row["label"] for row in selected_rows]),\n    snippet_offsets=np.cumsum([0] + steps_per_video),\n    score_columns=np.asarray(SCORE_COLUMNS),\n    scores=scores,\n    frame_gt=gt if not SMOKE_ONLY else np.empty(0, dtype=np.uint8),\n)\n(output_dir / "run.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")\nstage_directory.cleanup()\nprint("Đầu ra", output_dir)\n'
XD_RUNNER = '\nCOMMIT = "c41067f07d252efcda18008bea367886070c33b0"\nUPSTREAM_URL = "https://github.com/nwpu-zxr/VadCLIP.git"\nVIDEO_COUNT = 800\nFRAMES_PER_FEATURE = 16\nFRAME_COUNT = 2_331_296\nFEATURE_DIM = 512\nPAPER_AP = 0.8451\nPACK_MANIFEST_NAME = "xd_test_manifest.json"\nPACK_ARCHIVE_NAME = "vadclip_xd_test800.zip"\nPREVIOUS_PACK_ARCHIVE_NAME = "vadclip_xd_test800_kaggle_v2.zip"\nPACK_MANIFEST_SHA256 = "8e5d329708857338fa94fb63a1d5733d9d8536250e6c09c3ba012f54dce997b2"\nCLIP_SHA256 = "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f"\n\nimport csv\nimport hashlib\nimport importlib.util\nimport json\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nfrom collections import defaultdict\nfrom contextlib import ExitStack\nfrom pathlib import PurePosixPath\nfrom zipfile import BadZipFile, ZipFile\n\nif RUN_MODE not in {"smoke", "full"}:\n    raise ValueError("Evaluator XD cần chế độ smoke hoặc full.")\nif not INPUT.is_dir():\n    raise FileNotFoundError("Gắn dữ liệu qua Add Input trong Kaggle trước khi chạy.")\n\nOUTPUT.mkdir(parents=True, exist_ok=True)\nscratch = SRC_REPO.parent\nscratch.mkdir(parents=True, exist_ok=True)\nsrc_repo = SRC_REPO\nif not src_repo.exists():\n    with tempfile.TemporaryDirectory(dir=scratch) as checkout_dir:\n        checkout = Path(checkout_dir) / "source"\n        subprocess.run(["git", "clone", UPSTREAM_URL, str(checkout)], check=True)\n        subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", COMMIT], check=True)\n        checkout.rename(src_repo)\n\ndef git(*args):\n    return subprocess.check_output(["git", "-C", str(src_repo), *args], text=True).strip()\n\nif git("rev-parse", "HEAD") != COMMIT or git("status", "--porcelain", "--untracked-files=all"):\n    raise RuntimeError(f"Cần checkout sạch tại commit {COMMIT}. Kiểm tra {src_repo}.")\n\nnp = importlib.import_module("numpy")\n\ndef sha256(path):\n    with path.open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef zip_members(path):\n    try:\n        with ZipFile(path) as archive:\n            return [name for name in archive.namelist() if not name.endswith("/")]\n    except BadZipFile as exc:\n        raise ValueError(f"ZIP chưa tải đủ hoặc bị hỏng: {path.name}") from exc\n\n\ndef basename(value):\n    return PurePosixPath(str(value).replace(chr(92), "/")).name\n\n\ndef read_split(path):\n    with path.open(newline="", encoding="utf-8-sig") as stream:\n        reader = csv.DictReader(stream)\n        if reader.fieldnames is None or not {"path", "label"} <= set(reader.fieldnames):\n            raise ValueError("CSV tác giả cần cột path và label.")\n        fields, rows = reader.fieldnames, list(reader)\n    source_paths = [row["path"] for row in rows]\n    if (\n        len(rows) != VIDEO_COUNT\n        or len(set(source_paths)) != VIDEO_COUNT\n        or any(not path or not basename(path).endswith("__0.npy") for path in source_paths)\n        or any(not row["label"] for row in rows)\n    ):\n        raise ValueError(f"Split XD phải gồm {VIDEO_COUNT} dòng feature hợp lệ, duy nhất.")\n    return fields, rows\n\n\ndef matching_files(root, expected):\n    found = defaultdict(list)\n    if FEATURE_ZIP is None:\n        if not root.is_dir():\n            raise FileNotFoundError(f"DATA_ROOT không tồn tại: {root}")\n        for path in root.rglob("*.npy"):\n            if path.name in expected:\n                found[path.name].append((path, None))\n        if found:\n            return found\n        archives = sorted(root.rglob("*.zip"))\n    else:\n        archives = [Path(FEATURE_ZIP)]\n\n    for archive_path in archives:\n        if not archive_path.is_file():\n            raise FileNotFoundError(f"Không thấy FEATURE_ZIP: {archive_path}")\n        for member in zip_members(archive_path):\n            name = basename(member)\n            if name in expected:\n                found[name].append((archive_path, member))\n    return found\n\n\ndef match_one(original, candidates):\n    # Nếu hai file cùng tên, chỉ phân giải khi hậu tố thư mục chính thức khớp duy nhất.\n    if len(candidates) <= 1:\n        return candidates\n    official = PurePosixPath(original.replace(chr(92), "/")).parts\n    for depth in range(2, len(official) + 1):\n        matches = [\n            (path, member)\n            for path, member in candidates\n            if PurePosixPath((member or path.as_posix()).replace(chr(92), "/")).parts[-depth:]\n            == official[-depth:]\n        ]\n        if len(matches) == 1:\n            return matches\n        if not matches:\n            break\n    return candidates\n\n\ndef pick_checkpoint(root):\n    if CHECKPOINT is not None:\n        path = Path(CHECKPOINT)\n        if not path.is_file() or path.suffix.lower() not in {".pth", ".zip"}:\n            raise FileNotFoundError(f"CHECKPOINT cần đường dẫn .pth hoặc .zip hợp lệ: {path}")\n        return path\n    weights = [path for path in root.rglob("*.pth") if path.name in {"model_xd.pth", "model.pth"}]\n    if len(weights) == 1:\n        return weights[0]\n    if not weights:\n        archives = [\n            path for path in root.rglob("*.zip")\n            if any(\n                basename(member) in {"model_xd.pth", "model.pth"}\n                for member in zip_members(path)\n            )\n        ]\n        if len(archives) == 1:\n            return archives[0]\n    raise RuntimeError(\n        "Gắn checkpoint VadCLIP XD-Violence. Nếu tên khác hoặc có nhiều bản, "\n        "đặt CHECKPOINT bằng đường dẫn file .pth hoặc ZIP chỉ chứa một .pth."\n    )\n\n\ndef stage_member(archive, member, target):\n    # Chỉ ghi tới tên đích do notebook tạo. Không giải nén theo đường dẫn trong ZIP.\n    target.parent.mkdir(parents=True, exist_ok=True)\n    with archive.open(member) as src, target.open("wb") as dest:\n        shutil.copyfileobj(src, dest, length=1024 * 1024)\n    return target\n\n\ndef checkpoint_from_input(root, staging):\n    path = pick_checkpoint(root)\n    if path.suffix.lower() == ".zip":\n        members = [name for name in zip_members(path) if name.lower().endswith(".pth")]\n        if len(members) != 1:\n            raise ValueError(f"ZIP checkpoint phải chứa đúng một .pth: {path.name}")\n        if basename(members[0]) == "model_ucf.pth":\n            raise ValueError("ZIP đang chứa checkpoint UCF-Crime.")\n        with ZipFile(path) as archive:\n            path = stage_member(archive, members[0], staging / "checkpoint.pth")\n    if path.name == "model_ucf.pth" or path.stat().st_size == 0:\n        raise ValueError(f"File checkpoint không hợp lệ cho XD: {path.name}")\n    with path.open("rb") as stream:\n        if stream.read(4) == b"PK\\x03\\x04":\n            zip_members(path)  # xác nhận có thư mục ZIP ở cuối file PyTorch\n    return path\n\n\ndef feature_info(path):\n    array = np.load(path, mmap_mode="r", allow_pickle=False)\n    if (\n        array.ndim != 2 or array.shape[0] == 0 or array.shape[1] != FEATURE_DIM\n        or array.dtype not in (np.float16, np.float32, np.float64)\n        or not np.isfinite(array).all()\n    ):\n        raise ValueError(f"Feature cần mảng số thực hữu hạn [T, {FEATURE_DIM}]: {path.name}")\n    return len(array), sha256(path)\n\n\ndef read_pack_manifest(root):\n    candidates = [path.read_bytes() for path in root.rglob(PACK_MANIFEST_NAME)]\n    archives = set(root.rglob(PACK_ARCHIVE_NAME)) | set(root.rglob(PREVIOUS_PACK_ARCHIVE_NAME))\n    if FEATURE_ZIP is not None:\n        archives.add(Path(FEATURE_ZIP))\n    for archive_path in sorted(archives):\n        with ZipFile(archive_path) as archive:\n            members = [\n                name for name in archive.namelist() if basename(name) == PACK_MANIFEST_NAME\n            ]\n            candidates.extend(archive.read(name) for name in members)\n    if not candidates:\n        raise FileNotFoundError(\n            "Chưa thấy manifest. Gắn Dataset VadCLIP XD-Violence Test 800 "\n            f"từ {PACK_ARCHIVE_NAME} và giữ nguyên tên file ZIP."\n        )\n    if any(hashlib.sha256(raw).hexdigest() != PACK_MANIFEST_SHA256 for raw in candidates):\n        raise ValueError("Manifest trên Kaggle không khớp bản đã khóa trong notebook.")\n\n    manifest = json.loads(candidates[0])\n    if manifest["upstream_commit"] != COMMIT:\n        raise ValueError("Commit trong manifest không đúng.")\n    return manifest\n\n\ndef check_pack_manifest(manifest, inventory, checkpoint_hash):\n    if manifest["checkpoint_sha256"] != checkpoint_hash:\n        raise ValueError("Sai checkpoint so với manifest đã khóa.")\n    actual_features = {\n        basename(row["official_path"]): {"steps": row["steps"], "sha256": row["feature_sha256"]}\n        for row in inventory\n    }\n    if actual_features != manifest["features"]:\n        names = actual_features.keys() | manifest["features"].keys()\n        first_difference = next(\n            name for name in sorted(names)\n            if actual_features.get(name) != manifest["features"].get(name)\n        )\n        raise ValueError(f"Feature trên Kaggle lệch bản đã khóa: {first_difference}")\n    return manifest["source_features_sha256"]\n\n\ndef prepare_inputs(split, gt_path, data_root, staging, upload_names=None):\n    fields, rows = read_split(split)\n    original_names = {basename(row["path"]) for row in rows}\n    if upload_names is not None and (\n        len(original_names) != VIDEO_COUNT\n        or upload_names.keys() != original_names\n        or len(set(upload_names.values())) != VIDEO_COUNT\n    ):\n        raise ValueError("Ánh xạ tên file không khớp danh sách video trong CSV gốc.")\n    names = upload_names if upload_names is not None else {name: name for name in original_names}\n    found = matching_files(data_root, set(names.values()))\n    resolved = []\n    issues = {"missing": [], "ambiguous": []}\n    for row in rows:\n        original = row["path"]\n        candidates = match_one(original, found[names[basename(original)]])\n        if len(candidates) == 0:\n            issues["missing"].append(original)\n        elif len(candidates) > 1:\n            issues["ambiguous"].append({\n                "expected": original,\n                "candidates": [str(path) if member is None else f"{path.name}::{member}"\n                               for path, member in candidates],\n            })\n        else:\n            resolved.append(candidates[0])\n    if any(issues.values()) or len(set(resolved)) != VIDEO_COUNT:\n        report = OUTPUT / "input_issues.json"\n        report.write_text(json.dumps(issues, ensure_ascii=False, indent=2), encoding="utf-8")\n        raise RuntimeError(\n            f"Thiếu {len(issues[\'missing\'])} file, trùng {len(issues[\'ambiguous\'])} dòng "\n            f"hoặc dùng lại cùng feature. Xem {report}. Kiểm DATA_ROOT/FEATURE_ZIP."\n        )\n\n    gt = np.load(gt_path, allow_pickle=False)\n    if gt.shape != (FRAME_COUNT,) or not np.array_equal(np.unique(gt), [0, 1]):\n        raise ValueError("Ground truth XD sai shape hoặc không phải vector nhị phân.")\n\n    inventory = []\n    with ExitStack() as stack:\n        archives = {}\n        for index, (row, (path, member)) in enumerate(zip(rows, resolved, strict=True)):\n            if member is not None:\n                if path not in archives:\n                    archives[path] = stack.enter_context(ZipFile(path))\n                target = staging / f"{index:04d}" / basename(member)\n                path = stage_member(archives[path], member, target)\n            count, digest = feature_info(path)\n            inventory.append({\n                "official_path": row["path"], "label": row["label"],\n                "steps": count, "feature_sha256": digest,\n            })\n            row["path"] = str(path)\n    steps = sum(item["steps"] for item in inventory)\n    if steps * FRAMES_PER_FEATURE != len(gt):\n        raise ValueError(\n            f"Feature có {steps:,} bước, cần {len(gt) // FRAMES_PER_FEATURE:,} "\n            f"theo ground truth tác giả. Kiểm phiên bản gói feature và crop."\n        )\n\n    offset = 0\n    for item in inventory:\n        stop = offset + item["steps"] * FRAMES_PER_FEATURE\n        positive_frames = int(np.count_nonzero(gt[offset:stop]))\n        if "_label_A" in basename(item["official_path"]) and positive_frames:\n            raise ValueError(f"Lệch ground truth trong video Normal: {item[\'official_path\']}")\n        item["positive_frames"] = positive_frames\n        offset = stop\n\n    resolved_csv = staging / "xd_CLIP_rgbtest_kaggle.csv"\n    with resolved_csv.open("w", newline="", encoding="utf-8") as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows(rows)\n    return resolved_csv, inventory, gt\n\n# Preflight CPU. Mỗi lần chạy tạo thư mục mới, không ghi đè input đã gắn.\nstage_directory = tempfile.TemporaryDirectory(prefix="run_", dir=scratch)\nstaging = Path(stage_directory.name)\nsplit = src_repo / "list" / "xd_CLIP_rgbtest.csv"\ngt_path = src_repo / "list" / "gt.npy"\ndata_root = Path(DATA_ROOT) if DATA_ROOT is not None else INPUT\ntry:\n    pack_manifest = read_pack_manifest(INPUT) if REQUIRE_PACK_MANIFEST else None\n    upload_names = pack_manifest["upload_names"] if pack_manifest is not None else None\n    resolved_csv, inventory, gt = prepare_inputs(split, gt_path, data_root, staging, upload_names)\n    checkpoint_path = checkpoint_from_input(INPUT, staging)\n    checkpoint_hash = sha256(checkpoint_path)\n    source_features_hash = None\n    if pack_manifest is not None:\n        source_features_hash = check_pack_manifest(pack_manifest, inventory, checkpoint_hash)\nexcept Exception:\n    stage_directory.cleanup()\n    raise\n\nreceipt = {\n    "upstream_commit": COMMIT,\n    "split_sha256": sha256(split),\n    "ground_truth_sha256": sha256(gt_path),\n    "checkpoint_sha256": checkpoint_hash,\n    "pack_manifest_sha256": PACK_MANIFEST_SHA256 if REQUIRE_PACK_MANIFEST else None,\n    "source_features_archive_sha256": source_features_hash,\n    "video_count": len(inventory),\n    "snippet_count": sum(row["steps"] for row in inventory),\n    "frame_count": len(gt),\n    "frames_per_feature": FRAMES_PER_FEATURE,\n    "videos": inventory,\n}\nif REFERENCE_MANIFEST is not None:\n    try:\n        reference = json.loads(Path(REFERENCE_MANIFEST).read_text(encoding="utf-8"))\n        for key in ("upstream_commit", "split_sha256", "ground_truth_sha256",\n                    "checkpoint_sha256", "videos"):\n            if receipt[key] != reference.get(key):\n                raise ValueError(f"Bản Kaggle lệch bản tham chiếu ở trường {key}.")\n    except Exception:\n        stage_directory.cleanup()\n        raise\nreceipt["matched_reference_manifest"] = REFERENCE_MANIFEST is not None\n\n(OUTPUT / "preflight.json").write_text(\n    json.dumps(receipt, ensure_ascii=False, indent=2), encoding="utf-8"\n)\nprint(\n    f"Preflight đạt: {len(inventory)} video, {receipt[\'snippet_count\']:,} bước, "\n    f"{len(gt):,} frame."\n)\nprint(f"Checkpoint {checkpoint_path.name}, SHA-256 {checkpoint_hash}")\nif REQUIRE_PACK_MANIFEST:\n    print("SHA-256: khớp cả 800 feature và checkpoint công bố với hai gói đã chuẩn bị.")\n\nreceipt["evaluated_checkpoint_sha256"] = checkpoint_hash\n\nimport contextlib\nimport runpy\nimport time\nimport traceback\nfrom unittest.mock import patch\n\n\n\n\ndef run_upstream():\n    full_run = RUN_MODE == "full"\n    packages = {\n        "ftfy": "ftfy", "packaging": "packaging", "regex": "regex", "tqdm": "tqdm",\n        "sklearn": "scikit-learn", "scipy": "scipy", "pandas": "pandas",\n    }\n    missing = [\n        package for module, package in packages.items()\n        if importlib.util.find_spec(module) is None\n    ]\n    if missing:\n        subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)\n\n    import torch\n    from sklearn import metrics\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("Bật Accelerator GPU T4 trong Kaggle để chạy inference.")\n    torch.cuda.set_device(0)\n    torch.manual_seed(234)\n    runtime = staging / "runtime"\n    shutil.copytree(src_repo / "src", runtime)\n    from v7_windows import overlap_forward\n\n    clip_source = runtime / "clip" / "clip.py"\n    text = clip_source.read_text(encoding="utf-8")\n    old_import = "from pkg_resources import packaging"\n    if text.count(old_import) != 1:\n        raise RuntimeError("Không tìm thấy import CLIP cần sửa tương thích tại commit đã khóa.")\n    clip_source.write_text(text.replace(old_import, "import packaging.version"), encoding="utf-8")\n\n    if CLIP_WEIGHTS is not None:\n        source = Path(CLIP_WEIGHTS)\n        if not source.is_file() or sha256(source) != CLIP_SHA256:\n            raise ValueError("CLIP_WEIGHTS cần file OpenAI ViT-B-16.pt đúng SHA-256.")\n        cache = Path.home() / ".cache" / "clip"\n        cache.mkdir(parents=True, exist_ok=True)\n        target = cache / "ViT-B-16.pt"\n        if source.resolve() != target.resolve():\n            with tempfile.TemporaryDirectory(dir=cache) as copy_dir:\n                temporary = Path(copy_dir) / target.name\n                shutil.copyfile(source, temporary)\n                if sha256(temporary) != CLIP_SHA256:\n                    raise ValueError("Bản sao trọng số CLIP không khớp SHA-256.")\n                temporary.replace(target)\n\n    output_root = OUTPUT / RUN_MODE\n    output_root.mkdir(exist_ok=True)\n    output_dir = Path(tempfile.mkdtemp(prefix="run_", dir=output_root))\n    print(f"Đang chạy evaluator gốc. Log: {output_dir / \'upstream.log\'}", flush=True)\n    native_ap = metrics.average_precision_score\n    ap_calls = []\n    upstream_completed = False\n    post_ap_error = None\n    checkpoint_loaded = False\n    forwards = 0\n    torch_load = torch.load\n\n    class SmokeComplete(Exception):\n        pass\n\n    def load_checkpoint(file, *args, **kwargs):\n        nonlocal checkpoint_loaded\n        if not isinstance(file, (str, os.PathLike)) or Path(file).resolve() != checkpoint_path.resolve():\n            return torch_load(file, *args, **kwargs)\n        checkpoint_loaded = True\n        state = torch_load(file, map_location="cpu", weights_only=True)\n        for field in ("state_dict", "model_state_dict"):\n            if isinstance(state, dict) and field in state:\n                return state[field]\n        return state\n\n    def record_ap(y_true, y_score, *args, **kwargs):\n        value = native_ap(y_true, y_score, *args, **kwargs)\n        if np.asarray(y_true).shape == gt.shape and np.array_equal(y_true, gt):\n            scores = np.asarray(y_score).reshape(-1)\n            if scores.shape != gt.shape or not np.isfinite(scores).all():\n                raise ValueError("Điểm frame từ evaluator XD sai shape hoặc chứa NaN/Inf.")\n            grouped = scores.reshape(-1, FRAMES_PER_FEATURE)\n            if not np.all(grouped == grouped[:, :1]):\n                raise ValueError("Điểm từ evaluator không theo quy tắc lặp 16 frame.")\n            ap_calls.append((float(value), grouped[:, 0].copy()))\n        return value\n\n    original_cwd = Path.cwd()\n    original_path = sys.path.copy()\n    original_argv = sys.argv.copy()\n    original_bytecode = sys.dont_write_bytecode\n    try:\n        os.chdir(runtime)\n        sys.path.insert(0, str(runtime))\n        sys.dont_write_bytecode = True\n        from model import CLIPVAD\n        from utils.tools import get_batch_mask\n        from xd_option import parser\n\n        # Giữ các giá trị mặc định XD của tác giả, chỉ thay đường dẫn input.\n        destinations = {action.dest for action in parser._actions}\n        overrides = {\n            "test_list": str(resolved_csv),\n            "gt_path": str(gt_path),\n            "model_path": str(checkpoint_path),\n        }\n        defaults = parser.parse_args([])\n        for key in ("gt_segment_path", "gt_label_path"):\n            default = Path(getattr(defaults, key))\n            path = (src_repo / "src" / default).resolve()\n            if not path.is_file():\n                path = (src_repo / default).resolve()\n            relative = path.relative_to(src_repo)\n            git("ls-files", "--error-unmatch", relative.as_posix())\n            overrides[key] = str(path)\n        if not overrides.keys() <= destinations:\n            raise RuntimeError(f"CLI upstream thiếu tham số: {overrides.keys() - destinations}")\n        flags = []\n        for action in parser._actions:\n            if action.dest in overrides:\n                flags.extend((action.option_strings[0], overrides[action.dest]))\n\n        from head import apply_head, load_head\n\n        bundle = load_head(HEAD_PATH, STRENGTH, USE_TEMPORAL, torch.device("cuda:0")) if HEAD_PATH else None\n        original_forward = CLIPVAD.forward\n\n        smoke_forwards = max(3, next(\n            i for i, row in enumerate(inventory, 1) if row["steps"] > defaults.visual_length\n        ))\n\n        def count_forward(self, images, padding_mask, text, lengths):\n            nonlocal forwards\n\n            def forward(x, mask, prompts, sizes):\n                return original_forward(self, x, mask, prompts, sizes)\n\n            if VARIANT != "v0":\n                result = overlap_forward(forward, images, padding_mask, text, lengths, get_batch_mask)\n            else:\n                result = forward(images, padding_mask, text, lengths)\n            if bundle is not None:\n                t, c, a = result\n                c, a = apply_head(bundle, images, lengths, c, a, "A")\n                result = t, c, a\n            forwards += 1\n            if RUN_MODE == "smoke" and forwards == smoke_forwards:\n                if any(not torch.isfinite(tensor).all() for tensor in result):\n                    raise ValueError("Smoke test sinh NaN/Inf.")\n                raise SmokeComplete\n            return result\n\n        sys.argv = [str(runtime / "xd_test.py"), *flags]\n        torch.cuda.reset_peak_memory_stats()\n        started = time.perf_counter()\n        with (\n            (output_dir / "upstream.log").open("w", encoding="utf-8") as log,\n            contextlib.redirect_stdout(log),\n            contextlib.redirect_stderr(log),\n            patch.object(torch, "load", load_checkpoint),\n            patch.object(metrics, "average_precision_score", record_ap),\n            patch.object(CLIPVAD, "forward", count_forward),\n        ):\n            try:\n                runpy.run_path(str(runtime / "xd_test.py"), run_name="__main__")\n                upstream_completed = True\n            except SmokeComplete:\n                if RUN_MODE != "smoke":\n                    raise\n            except Exception as exc:\n                if not full_run or len(ap_calls) != 2:\n                    raise\n                traceback.print_exc(file=log)\n                post_ap_error = f"{type(exc).__name__}: {exc}"\n        elapsed = time.perf_counter() - started\n    finally:\n        os.chdir(original_cwd)\n        sys.argv = original_argv\n        sys.path[:] = original_path\n        sys.dont_write_bytecode = original_bytecode\n\n    if not checkpoint_loaded or forwards == 0:\n        raise RuntimeError("Chưa xác nhận upstream đã nạp checkpoint XD và chạy forward.")\n\n    result = {\n        **receipt,\n        "run_kind": "smoke_inference" if RUN_MODE == "smoke" else "full_checkpoint_inference",\n        "runtime_patch": "CLIP imports packaging.version directly",\n        "clip_backbone_sha256": CLIP_SHA256,\n        "forward_calls": forwards,\n        "visual_block_length": defaults.visual_length,\n        "upstream_completed": upstream_completed,\n        "post_ap_error": post_ap_error,\n        "gpu": torch.cuda.get_device_name(0),\n        "torch": torch.__version__,\n        "elapsed_seconds": elapsed,\n        "peak_torch_vram_mib": torch.cuda.max_memory_allocated() / 2**20,\n        "metrics": None,\n    }\n    if full_run:\n        if len(ap_calls) != 2:\n            raise RuntimeError(\n                f"Cần hai AP cấp frame từ xd_test.py, nhận {len(ap_calls)}. "\n                f"Xem {output_dir / \'upstream.log\'}."\n            )\n        scores = np.column_stack([column for _, column in ap_calls])\n        result["paper_table_1_frame_ap"] = PAPER_AP\n        result["metrics"] = {\n            f"branch_{index + 1}": {\n                "frame_ap": ap,\n                "roc_auc": float(metrics.roc_auc_score(gt, np.repeat(column, FRAMES_PER_FEATURE))),\n                "gap_vs_paper_percentage_points": 100 * (ap - PAPER_AP),\n            }\n            for index, (ap, column) in enumerate(ap_calls)\n        }\n        score_columns = ["branch_1", "branch_2"]\n        steps_per_video = [item["steps"] for item in inventory]\n        result["score_columns"] = score_columns\n        np.savez_compressed(\n            output_dir / "predictions.npz",\n            official_path=np.asarray([item["official_path"] for item in inventory]),\n            video_label=np.asarray([item["label"] for item in inventory]),\n            snippet_offsets=np.cumsum([0] + steps_per_video),\n            score_columns=np.asarray(score_columns),\n            scores=scores,\n            frame_gt=gt,\n        )\n    elif forwards != smoke_forwards:\n        raise RuntimeError("Smoke test chưa đủ lượt forward.")\n\n    (output_dir / "run.json").write_text(\n        json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    if full_run:\n        if post_ap_error is not None:\n            print(f"Script gốc lỗi sau khi tính đủ hai AP. Xem upstream.log: {post_ap_error}")\n        for name, values in result["metrics"].items():\n            print(\n                f"{name}: AP {values[\'frame_ap\'] * 100:.4f}% "\n                f"(chênh {values[\'gap_vs_paper_percentage_points\']:+.4f} điểm phần trăm)"\n            )\n    else:\n        print("Smoke đạt, gồm video dài hơn block.")\n    print(f"Đầu ra: {output_dir}")\n\n\ntry:\n    if RUN_MODE == "preflight":\n        print("Đã kiểm input. Chọn GPU và RUN_MODE = \'smoke\' để thử checkpoint.")\n    else:\n        run_upstream()\nfinally:\n    stage_directory.cleanup()\n'


In [ ]:
import csv
import hashlib
import importlib.util
import json
import math
import os
import shutil
import subprocess
import sys
import tempfile
from collections import deque
from zipfile import ZIP_DEFLATED, ZipFile


def runner_header(config):
    source = ["from pathlib import Path", ""]
    for key, value in config.items():
        code = f"Path({str(value)!r})" if isinstance(value, Path) else repr(value)
        source.append(f"{key} = {code}")
    return "\n".join(source)


def run_dataset(name, runner, config, work_dir):
    script = work_dir / f"{name}.py"
    script.write_text(
        runner_header(config) + "\n\n" + runner,
        encoding="utf-8",
    )
    console_path = work_dir.parent / f"{name}_console.log"
    print(f"\n--- {name} ---", flush=True)
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    with (
        console_path.open("w", encoding="utf-8") as console,
        subprocess.Popen(
            [sys.executable, "-u", str(script)],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            encoding="utf-8", errors="replace", env=env,
        ) as process,
    ):
        for line in process.stdout:
            console.write(line)
            print(line, end="", flush=True)
        if process.wait() != 0:
            raise RuntimeError(f"Tiến trình {name} dừng với mã {process.returncode}.")

V0_REFERENCE = {
    "ucf": {"roc_auc": 0.8801992605166816, "ano_auc": 0.7023225660564671, "ap": 0.3355725526808031},
    "xd": {"frame_ap": 0.8450448809004643, "roc_auc": 0.946812650099563},
}


def file_sha(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


V0_CHECKPOINT_SHA = {
    "ucf": "d064aba05190ce5f6652151e6259b24ca6dda83a1e6289bf343afb6f905a2c79",
    "xd": "a676c14d794a7046f7636041e601a5d9a2bef03708dbc4cae90f8b37ac2af2b4",
}


def resolve_checkpoint(prefix, override, directory):
    """Return the pinned V0 file, verifying bytes before torch.load can see them."""
    if override is not None:
        candidates = [Path(override)]
    else:
        files = {f"model_{prefix}.pth"}
        if prefix == "xd":
            files.add("model.pth")
        candidates = sorted(
            path for path in INPUT.rglob("*")
            if path.is_file() and (
                path.name.lower() in files
                or (path.suffix.lower() == ".zip" and prefix in path.stem.lower()
                    and "checkpoint" in path.stem.lower())
            )
        )
    expected = V0_CHECKPOINT_SHA[prefix]
    for path in candidates:
        if not path.is_file():
            continue
        if path.suffix.lower() == ".pth" and file_sha(path) == expected:
            return path
        if path.suffix.lower() != ".zip":
            continue
        with ZipFile(path) as archive:
            for member in archive.infolist():
                if member.is_dir() or not member.filename.lower().endswith(".pth"):
                    continue
                with archive.open(member) as source:
                    digest = hashlib.file_digest(source, "sha256").hexdigest()
                if digest != expected:
                    continue
                directory.mkdir(parents=True, exist_ok=True)
                target = directory / f"model_{prefix}.pth"
                with archive.open(member) as source, target.open("wb") as destination:
                    shutil.copyfileobj(source, destination)
                if file_sha(target) != expected:
                    raise ValueError(f"{prefix}: staged checkpoint SHA-256 mismatch.")
                return target
    raise FileNotFoundError(
        f"{prefix}: attach the original V0 checkpoint or set {prefix.upper()}_CHECKPOINT "
        "to its .pth or ZIP. Candidates must match the pinned V0 SHA-256."
    )


def format_table(headers, values):
    widths = [max(map(len, column)) for column in zip(headers, *values, strict=True)]

    def line(cells):
        return "| " + " | ".join(
            value.ljust(width) for value, width in zip(cells, widths, strict=True)
        ) + " |"

    return "\n".join([line(headers), line(["-" * width for width in widths]),
                      *(line(row) for row in values)])


def comparison_report(rows):
    datasets = {"ucf": "UCF-Crime", "xd": "XD-Violence"}
    metrics = {"roc_auc": "ROC-AUC", "ap": "Frame AP", "frame_ap": "Frame AP",
               "ano_auc": "Ano-AUC"}
    variants = {"v7": "V7", "clip_only": "CLIP", "temporal_random": "CLIP + temporal", "v18": "V18"}
    scores, differences = {}, []
    for row in rows:
        dataset, branch, metric = row["dataset"], row["branch"], row["metric"]
        label = f'{metrics[metric]}{" *" if row["primary"] else ""}'
        key = (datasets[dataset], branch, label)
        scores.setdefault(key, {}).update({
            row["control"]: row["control_pct"], row["treatment"]: row["treatment_pct"],
        })
        if row["primary"] or (row["comparison"] == "v18_vs_v7" and branch == "A"):
            low, high = row["ci95_low_pp"], row["ci95_high_pp"]
            bounds = "—" if low is None or high is None else f"[{low:+.5f}, {high:+.5f}]"
            pair = f'{variants[row["treatment"]]} − {variants[row["control"]]}'
            differences.append([*key, pair, f'{row["delta_pp"]:+.5f}', bounds])

    score_rows = [[*key, *(f"{values[v]:.5f}%" if v in values else "—" for v in VARIANTS)]
                  for key, values in scores.items()]
    return "\n\n".join([
        "## Điểm theo cấu hình",
        format_table(["Tập dữ liệu", "Nhánh", "Chỉ số", *variants.values()], score_rows),
        "## Chênh lệch so với đối chứng",
        format_table(["Tập dữ liệu", "Nhánh", "Chỉ số", "So sánh", "Δ (đpt)", "CI 95% (đpt)"],
                     differences),
        "* Chỉ số chính của tập dữ liệu. CI bootstrap ghép cặp theo video. "
        "Các đối chiếu đầy đủ nằm trong comparison.csv.",
    ])


# ruff: noqa: F821
"""Kaggle notebook entry point. Earlier cells embed the evaluators and config."""

import time

VARIANTS = ("v7", "clip_only", "temporal_random", "v18")
PAIRS = (("v7", "clip_only"), ("clip_only", "temporal_random"),
         ("temporal_random", "v18"), ("v7", "v18"))


def compare_variants(prefix, paths, analysis):
    import numpy as np

    arrays, reports = {}, {}
    for name, path in paths.items():
        reports[name] = json.loads(path.read_text(encoding="utf-8"))
        with np.load(path.with_name("predictions.npz"), allow_pickle=False) as data:
            arrays[name] = dict(data)
    baseline = reports["v7"]
    base = arrays["v7"]
    id_key = "video_id" if prefix == "ucf" else "official_path"
    offsets = base["snippet_offsets"]
    gt = base["frame_gt"].reshape(-1)
    expected = 290 if prefix == "ucf" else 800
    if len(gt) != offsets[-1] * 16 or len(offsets) != expected + 1:
        raise ValueError("Invalid official frame grid.")
    primary_column = 0 if prefix == "ucf" else 1
    primary_branch = "classification" if prefix == "ucf" else "branch_2"
    primary_metric = "roc_auc" if prefix == "ucf" else "frame_ap"
    target = 0.8802335993846885 if prefix == "ucf" else 0.8461979894100824
    if not abs(baseline["metrics"][primary_branch][primary_metric] - target) <= 0.00005:
        raise ValueError("V7 reproduction check failed.")
    for name, current in arrays.items():
        report = reports[name]
        keys = ("split_sha256", "ground_truth_sha256", "evaluated_checkpoint_sha256",
                "frames_per_feature", "visual_block_length")
        if (report["run_kind"] != "full_checkpoint_inference" or report["video_count"] != expected
                or any(report[k] != baseline[k] for k in keys)):
            raise ValueError(f"Wrong checkpoint or evaluation grid: {prefix}/{name}")
        if (current["scores"].shape != (offsets[-1], 2) or
                not np.isfinite(current["scores"]).all() or
                any(not np.array_equal(current[k], base[k]) for k in
                    (id_key, "snippet_offsets", "frame_gt", "score_columns", "video_label"))):
            raise ValueError(f"Mismatched evaluator outputs: {prefix}/{name}")
        if not np.array_equal(current["scores"][:, 1-primary_column],
                              base["scores"][:, 1-primary_column]):
            raise ValueError(f"Unexpected change to the secondary branch: {prefix}/{name}")

    rows = []
    names = ("classification", "alignment") if prefix == "ucf" else ("branch_1", "branch_2")
    for control, treatment in PAIRS:
        for col, branch in enumerate(names):
            c = arrays[control]["scores"][:, col]
            t = arrays[treatment]["scores"][:, col]
            boot = analysis.paired_bootstrap(c, t, gt, offsets, BOOTSTRAP_REPLICATES, SEED)
            for metric in reports[control]["metrics"][branch]:
                if metric not in {"roc_auc", "ap", "frame_ap", "ano_auc"}:
                    continue
                a, b = (reports[v]["metrics"][branch][metric] * 100 for v in (control, treatment))
                key = {"roc_auc": "auc_ci95_pp", "ap": "ap_ci95_pp", "frame_ap": "ap_ci95_pp"}
                low, high = boot.get(key.get(metric)) or (None, None)
                rows.append({
                    "dataset": prefix, "branch": "CA"[col], "metric": metric,
                    "primary": col == primary_column and metric == primary_metric,
                    "comparison": f"{treatment}_vs_{control}", "control": control,
                    "treatment": treatment, "control_pct": a, "treatment_pct": b, "delta_pp": b-a,
                    "ci95_low_pp": low, "ci95_high_pp": high,
                })
    coverage = analysis.compare_coverage(
        prefix, paths, reports, (0.01, 0.05), SEED, BOOTSTRAP_REPLICATES, "v7", "v18",
    )
    return rows, coverage


def report_path(destination):
    files = list((destination / RUN_MODE).rglob("run.json"))
    if len(files) != 1:
        raise ValueError(f"Expected one evaluator report: {destination}")
    return files[0]


def run_both():
    import numpy as np

    if RUN_MODE not in {"smoke", "full"} or not DATASETS:
        raise ValueError("Use RUN_MODE='smoke' or 'full' and a nonempty DATASETS.")
    if len(set(DATASETS)) != len(DATASETS) or set(DATASETS) - {"ucf", "xd"}:
        raise ValueError("DATASETS accepts ucf and xd without duplicates.")
    if BOOTSTRAP_REPLICATES < 1 or not INPUT.is_dir():
        raise ValueError("Check BOOTSTRAP_REPLICATES and Kaggle Add Input.")
    root = WORK / "vadclip_v18"
    root.mkdir(parents=True, exist_ok=True)
    output = Path(tempfile.mkdtemp(prefix="run_", dir=root))
    scripts = output / "runtime_scripts"
    scripts.mkdir()
    for name, source in SOURCES.items():
        (scripts / name).write_text(source, encoding="utf-8")
    spec = importlib.util.spec_from_file_location("v18_analysis", scripts / "analysis.py")
    analysis = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(analysis)
    shared = {
        "INPUT": INPUT, "WORK": WORK, "CLIP_WEIGHTS": CLIP_WEIGHTS,
        "SMOKE_ONLY": RUN_MODE == "smoke",
        "SRC_REPO": Path(tempfile.gettempdir()) / "vadclip_xd_c41067f" / "upstream",
    }
    jobs = {
        "ucf": (UCF_RUNNER, UCF_TRAIN_ROOT, UCF_CHECKPOINT, {"DATA_ROOT": UCF_DATA_ROOT}),
        "xd": (XD_RUNNER, XD_TRAIN_ROOT, XD_CHECKPOINT, {
            "DATA_ROOT": XD_DATA_ROOT, "FEATURE_ZIP": XD_FEATURE_ZIP, "REFERENCE_MANIFEST": None,
            "REQUIRE_PACK_MANIFEST": True, "RUN_MODE": RUN_MODE,
        }),
    }
    started = time.perf_counter()
    results, rows = {}, []
    experiment = {
        "version": "v18", "mode": RUN_MODE, "seed": SEED, "bootstrap": BOOTSTRAP_REPLICATES,
        "training": TEMPORAL_CONFIG,
        "source_sha256": {name: hashlib.sha256(code.encode()).hexdigest() for name, code in
                          {**SOURCES, "trainer": TRAINER_SOURCE, "ucf": UCF_RUNNER, "xd": XD_RUNNER}.items()},
    }
    (output / "experiment.json").write_text(json.dumps(experiment, indent=2), encoding="utf-8")
    print("Output:", output, flush=True)
    for prefix in DATASETS:
        runner, train_root, checkpoint_override, config = jobs[prefix]
        entry = results[prefix] = {"status": "running", "reports": {}}
        train_output = output / prefix / "train"
        paths = {}
        try:
            checkpoint = resolve_checkpoint(prefix, checkpoint_override, output / prefix / "v0_input")
            evaluation = {**shared, **config, "CHECKPOINT": checkpoint}
            dest = output / prefix / "v7"
            run_dataset(f"{prefix}_v7", runner, {
                **evaluation, "OUTPUT": dest, "VARIANT": "v7",
                "HEAD_PATH": None, "STRENGTH": 0., "USE_TEMPORAL": False,
            }, scripts)
            paths["v7"] = report_path(dest)
            entry["reports"]["v7"] = paths["v7"].relative_to(output).as_posix()
            run_dataset(f"{prefix}_train", TRAINER_SOURCE, {
                **shared, "OUTPUT": train_output, "PREFIX": prefix, "TRAIN_ROOT": train_root,
                "V0_PATH": checkpoint, "V0_SHA256": V0_CHECKPOINT_SHA[prefix],
                "MANIFEST_SHA256": TRAIN_MANIFESTS[prefix], "SEED": SEED, "HALF_BATCH": HALF_BATCH,
                "VARIANTS": VARIANTS[1:], "TEMPORAL_CONFIG": TEMPORAL_CONFIG,
            }, scripts)
            trained = json.loads((train_output / "training.json").read_text(encoding="utf-8"))
            entry["training"] = (train_output / "training.json").relative_to(output).as_posix()
            for variant in VARIANTS[1:]:
                record = trained["variants"][variant]
                weights = train_output / f"{variant}.pth"
                if file_sha(weights) != record["checkpoint_sha256"]:
                    raise ValueError(f"Head checkpoint hash mismatch: {prefix}/{variant}")
                dest = output / prefix / variant
                run_dataset(f"{prefix}_{variant}", runner, {
                    **evaluation, "OUTPUT": dest, "VARIANT": variant, "HEAD_PATH": weights,
                    "STRENGTH": 1. if RUN_MODE == "smoke" else record["strength"],
                    "USE_TEMPORAL": variant != "clip_only",
                }, scripts)
                paths[variant] = report_path(dest)
                entry["reports"][variant] = paths[variant].relative_to(output).as_posix()
            if RUN_MODE == "full":
                metrics, entry["coverage_v18_vs_v7"] = compare_variants(prefix, paths, analysis)
                rows.extend(metrics)
                column = 0 if prefix == "ucf" else 1
                entry["false_alarm_at_recall"] = {}
                for variant in ("v7", "v18"):
                    with np.load(paths[variant].with_name("predictions.npz"), allow_pickle=False) as data:
                        entry["false_alarm_at_recall"][variant] = analysis.false_alarm_diagnostics(
                            data["scores"][:, column], data["frame_gt"].reshape(-1), data["snippet_offsets"],
                        )
            entry["status"] = "ok"
        except (OSError, ValueError, KeyError, RuntimeError) as exc:
            entry.update(status="failed", error=f"{type(exc).__name__}: {exc}")
            print(prefix, entry["error"], flush=True)
        finally:
            (output / "summary.json").write_text(json.dumps({
                **experiment, "runs": results, "metrics": rows,
                "elapsed_seconds": time.perf_counter() - started,
            }, ensure_ascii=False, indent=2), encoding="utf-8")
    if rows:
        with (output / "comparison.csv").open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=rows[0])
            writer.writeheader()
            writer.writerows(rows)
        report = comparison_report(rows)
        (output / "comparison.md").write_text(report + "\n", encoding="utf-8")
        print(report, flush=True)
    archive = output / "vadclip_v18_results.zip"
    with ZipFile(archive, "w", ZIP_DEFLATED) as handle:
        for path in sorted(output.rglob("*")):
            if path.is_file() and (
                path.suffix in {".json", ".csv", ".md", ".log", ".py"}
                or path.suffix == ".pth" and path.parent.name == "train"
                or INCLUDE_PREDICTIONS and path.name == "predictions.npz"
            ):
                handle.write(path, path.relative_to(output))
    print("Download:", archive, flush=True)
    if any(entry["status"] != "ok" for entry in results.values()):
        raise RuntimeError("Lượt chạy thiếu kết quả. Xem summary.json và console log.")
    return archive


RESULT_ZIP = run_both()
